# Imports

In [ ]:
#!pip install torch_geometric

In [ ]:
#!pip install pandas

In [ ]:
#!pip install numpy scipy  matplotlib scikit-learn networkx tqdm

In [ ]:
#!pip install graphlime

In [1]:
import torch
import sys


from torch_geometric.datasets import Planetoid

c:\faculdade\Tabalho-Final-XAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from torch_geometric.datasets import DBLP
import os
import pandas as pd

In [3]:
from graphlime import GraphLIME


In [ ]:
from itertools import combinations
from tqdm import tqdm
import re
from scipy.stats import spearmanr
from torch_geometric.explain import (
    Explainer,
    GNNExplainer
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
import numpy as np
import warnings
import pickle
from pathlib import Path

# Dados

In [4]:
def load_dataset(name):

    if name in [
        "Cora",
        "CiteSeer",
        "PubMed"
    ]:

        dataset = Planetoid(
            root=f"data/{name}",
            name=name
        )

        return dataset

    elif name == "DBLP":

        dataset = DBLP(
            root="data/DBLP"
        )

        return dataset

    else:

        raise ValueError(
            f"Dataset {name} não suportado."
        )

Cora

In [5]:
PLdataset = Planetoid(root='data/Planetoid', name='Cora')
cora = PLdataset[0]

print(cora)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [6]:
print(f"Nós: {cora.num_nodes}")
print(f"Arestas: {cora.num_edges}")
print(f"Features: {cora.num_features}")
print(f"Classes: {PLdataset.num_classes}")

Nós: 2708
Arestas: 10556
Features: 1433
Classes: 7


In [112]:
class_counts = torch.bincount(cora.y)

for i, count in enumerate(class_counts):
    print(f"Classe {i}: {count}")

Classe 0: 351
Classe 1: 217
Classe 2: 418
Classe 3: 818
Classe 4: 426
Classe 5: 298
Classe 6: 180


In [113]:
def load_cora():
    dataset = Planetoid(
        root='data/Planetoid',
        name='Cora'
    )

    return dataset[0]

In [114]:
random.seed(42)

candidate_nodes = [
    i for i in range(cora.num_nodes)
    if cora.test_mask[i]
]

selected_nodes = random.sample(candidate_nodes, 20)

print(selected_nodes)

[2362, 1822, 1733, 2467, 1989, 1958, 1936, 1850, 2462, 1812, 2400, 2466, 2621, 2266, 1797, 2312, 2140, 1740, 1738, 1803]


CitesSeer

In [7]:
CSdataset = Planetoid(
    root='data/Planetoid',
    name='CiteSeer'
)

In [8]:
citeseer = CSdataset[0]

print(citeseer)

Data(x=[3327, 3703], edge_index=[2, 9104], y=[3327], train_mask=[3327], val_mask=[3327], test_mask=[3327])


In [117]:
print(f"Nós: {citeseer.num_nodes}")
print(f"Arestas: {citeseer.num_edges}")
print(f"Features: {citeseer.num_features}")
print(f"Classes: {CSdataset.num_classes}")

Nós: 3327
Arestas: 9104
Features: 3703
Classes: 6


PubMed

In [9]:
MPdataset = Planetoid(
    root='data/Planetoid',
    name='PubMed'
)

In [10]:
pubmed = MPdataset[0]

print(pubmed)

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])


In [120]:
print(f"Nós: {pubmed.num_nodes}")
print(f"Arestas: {pubmed.num_edges}")
print(f"Features: {pubmed.num_features}")
print(f"Classes: {MPdataset.num_classes}")

Nós: 19717
Arestas: 88648
Features: 500
Classes: 3


DBLP

In [121]:
DBdataset = DBLP(
    root='data/DBLP'
)

KeyboardInterrupt: 

In [ ]:
dblp = DBdataset[0]

print(dblp)

HeteroData(
  author={
    x=[4057, 334],
    y=[4057],
    train_mask=[4057],
    val_mask=[4057],
    test_mask=[4057],
  },
  paper={ x=[14328, 4231] },
  term={ x=[7723, 50] },
  conference={ num_nodes=20 },
  (author, to, paper)={ edge_index=[2, 19645] },
  (paper, to, author)={ edge_index=[2, 19645] },
  (paper, to, term)={ edge_index=[2, 85810] },
  (paper, to, conference)={ edge_index=[2, 14328] },
  (term, to, paper)={ edge_index=[2, 85810] },
  (conference, to, paper)={ edge_index=[2, 14328] }
)


# Model

In [12]:
seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GCN

In [13]:
class GCN(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout=0.5
    ):
        super().__init__()

        self.conv1 = GCNConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = GCNConv(
            hidden_channels,
            out_channels
        )

        self.dropout = dropout

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

GAT

In [14]:
class GAT(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        heads=8,
        dropout=0.6
    ):
        super().__init__()

        self.gat1 = GATConv(
            in_channels,
            hidden_channels,
            heads=heads,
            dropout=dropout
        )

        self.gat2 = GATConv(
            hidden_channels * heads,
            out_channels,
            heads=1,
            concat=False,
            dropout=dropout
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        edge_index
    ):

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.gat1(
            x,
            edge_index
        )

        x = F.elu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.gat2(
            x,
            edge_index
        )

        return x

GraphSage

In [15]:
class GraphSAGE(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout=0.5
    ):
        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = SAGEConv(
            hidden_channels,
            out_channels
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        edge_index
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index
        )

        return x

funções de apoio

In [16]:
def build_model(
    model_name,
    in_channels,
    out_channels,
    config
):

    hidden_channels = config["hidden_channels"]

    if model_name == "GCN":

        return GCN(
            in_channels=in_channels,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            dropout=config["dropout"]
        )

    elif model_name == "GAT":

        return GAT(
            in_channels=in_channels,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            heads=config["heads"],
            dropout=config["dropout"]
        )

    elif model_name == "GraphSAGE":

        return GraphSAGE(
            in_channels=in_channels,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            dropout=config["dropout"]
        )

    else:

        raise ValueError(
            f"Modelo {model_name} não suportado."
        )

In [17]:
from itertools import product


def generate_model_variants(model_name):

    variants = []

    if model_name == "GCN":

        hidden_sizes = [16, 32, 64]
        dropouts = [0.3, 0.5, 0.7]

        for hidden, dropout in product(
            hidden_sizes,
            dropouts
        ):

            variants.append({
                "hidden_channels": hidden,
                "dropout": dropout
            })

    elif model_name == "GAT":

        hidden_sizes = [8, 16, 32]
        heads_list = [4, 8]
        dropouts = [0.4, 0.6]

        for hidden, heads, dropout in product(
            hidden_sizes,
            heads_list,
            dropouts
        ):

            variants.append({
                "hidden_channels": hidden,
                "heads": heads,
                "dropout": dropout
            })

    elif model_name == "GraphSAGE":

        hidden_sizes = [16, 32, 64]
        dropouts = [0.3, 0.5, 0.7]

        for hidden, dropout in product(
            hidden_sizes,
            dropouts
        ):

            variants.append({
                "hidden_channels": hidden,
                "dropout": dropout
            })

    else:

        raise ValueError(
            f"Modelo '{model_name}' não suportado."
        )

    return variants

# Treinamento

In [18]:
def train(model,data,optimizer):

    model.train()

    optimizer.zero_grad()

    out = model(
        data.x,
        data.edge_index
    )

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()

    optimizer.step()

    return loss.item()

In [19]:
@torch.no_grad()
def evaluate(model,data):

    model.eval()

    out = model(
        data.x,
        data.edge_index
    )

    pred = out.argmax(dim=1)

    train_acc = (
        pred[data.train_mask]
        == data.y[data.train_mask]
    ).float().mean().item()

    val_acc = (
        pred[data.val_mask]
        == data.y[data.val_mask]
    ).float().mean().item()

    test_acc = (
        pred[data.test_mask]
        == data.y[data.test_mask]
    ).float().mean().item()

    return train_acc, val_acc, test_acc

## GCN Baseline

Cora

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

cora = Planetoid(
    root="data/Planetoid",
    name="Cora"
)

Coradata = cora[0].to(device)

gcn_base_cora = GCN(
    in_channels=cora.num_features,
    hidden_channels=16,
    out_channels=cora.num_classes
).to(device)

optimizer = torch.optim.Adam(
    gcn_base_cora.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(gcn_base_cora,Coradata)

    train_acc, val_acc, test_acc = evaluate(gcn_base_cora,Coradata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.7552 | Train 0.9857 | Val 0.7920 | Test 0.7960
Epoch 020 | Loss 0.2232 | Train 1.0000 | Val 0.7900 | Test 0.7990
Epoch 030 | Loss 0.0896 | Train 1.0000 | Val 0.7820 | Test 0.7940
Epoch 040 | Loss 0.0446 | Train 1.0000 | Val 0.7860 | Test 0.7980
Epoch 050 | Loss 0.0313 | Train 1.0000 | Val 0.7840 | Test 0.7930
Epoch 060 | Loss 0.0382 | Train 1.0000 | Val 0.7860 | Test 0.7920
Epoch 070 | Loss 0.0380 | Train 1.0000 | Val 0.7840 | Test 0.7990
Epoch 080 | Loss 0.0423 | Train 1.0000 | Val 0.7680 | Test 0.7880
Epoch 090 | Loss 0.0294 | Train 1.0000 | Val 0.7780 | Test 0.8070
Epoch 100 | Loss 0.0408 | Train 1.0000 | Val 0.7660 | Test 0.7920
Epoch 110 | Loss 0.0418 | Train 1.0000 | Val 0.7840 | Test 0.8120
Epoch 120 | Loss 0.0275 | Train 1.0000 | Val 0.7820 | Test 0.8050
Epoch 130 | Loss 0.0206 | Train 1.0000 | Val 0.7680 | Test 0.8030
Epoch 140 | Loss 0.0328 | Train 1.0000 | Val 0.7740 | Test 0.8050
Epoch 150 | Loss 0.0263 | Train 1.0000 | Val 0.7800 | Test 0.8160
Epoch 160 

CiteSeer

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

citeseer = Planetoid(
    root="data/Planetoid",
    name="CiteSeer"
)

citeseerdata = citeseer[0].to(device)

gcn_base_citeseer = GCN(
    in_channels=citeseer.num_features,
    hidden_channels=16,
    out_channels=citeseer.num_classes
).to(device)

optimizer = torch.optim.Adam(
    gcn_base_citeseer.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

Processing...
Done!


In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(gcn_base_citeseer,citeseerdata)

    train_acc, val_acc, test_acc = evaluate(gcn_base_citeseer,citeseerdata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.4224 | Train 0.9750 | Val 0.6660 | Test 0.6620
Epoch 020 | Loss 0.1463 | Train 1.0000 | Val 0.6680 | Test 0.6640
Epoch 030 | Loss 0.0751 | Train 1.0000 | Val 0.6600 | Test 0.6590
Epoch 040 | Loss 0.0540 | Train 1.0000 | Val 0.6600 | Test 0.6590
Epoch 050 | Loss 0.0445 | Train 1.0000 | Val 0.6600 | Test 0.6620
Epoch 060 | Loss 0.0366 | Train 1.0000 | Val 0.6700 | Test 0.6690
Epoch 070 | Loss 0.0257 | Train 1.0000 | Val 0.6780 | Test 0.6480
Epoch 080 | Loss 0.0323 | Train 1.0000 | Val 0.6800 | Test 0.6750
Epoch 090 | Loss 0.0204 | Train 1.0000 | Val 0.6880 | Test 0.6720
Epoch 100 | Loss 0.0418 | Train 1.0000 | Val 0.6840 | Test 0.6790
Epoch 110 | Loss 0.0208 | Train 1.0000 | Val 0.6880 | Test 0.6730
Epoch 120 | Loss 0.0168 | Train 1.0000 | Val 0.6940 | Test 0.6720
Epoch 130 | Loss 0.0308 | Train 1.0000 | Val 0.6700 | Test 0.6650
Epoch 140 | Loss 0.0450 | Train 1.0000 | Val 0.6980 | Test 0.6790
Epoch 150 | Loss 0.0136 | Train 1.0000 | Val 0.6820 | Test 0.6770
Epoch 160 

PubMed

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

pubmed = Planetoid(
    root="data/Planetoid",
    name="PubMed"
)

pubmeddata = pubmed[0].to(device)

gcn_base_pubmed = GCN(
    in_channels=pubmed.num_features,
    hidden_channels=16,
    out_channels=pubmed.num_classes
).to(device)

optimizer = torch.optim.Adam(
    gcn_base_pubmed.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

Processing...
Done!


In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(gcn_base_pubmed,pubmeddata)

    train_acc, val_acc, test_acc = evaluate(gcn_base_pubmed,pubmeddata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.9280 | Train 0.9333 | Val 0.7300 | Test 0.6950
Epoch 020 | Loss 0.6786 | Train 0.9333 | Val 0.7440 | Test 0.7350
Epoch 030 | Loss 0.4297 | Train 0.9500 | Val 0.7620 | Test 0.7480
Epoch 040 | Loss 0.3438 | Train 0.9667 | Val 0.7540 | Test 0.7460
Epoch 050 | Loss 0.2841 | Train 1.0000 | Val 0.7800 | Test 0.7700
Epoch 060 | Loss 0.2008 | Train 1.0000 | Val 0.7800 | Test 0.7710
Epoch 070 | Loss 0.1629 | Train 1.0000 | Val 0.7880 | Test 0.7830
Epoch 080 | Loss 0.1411 | Train 1.0000 | Val 0.7900 | Test 0.7870
Epoch 090 | Loss 0.1527 | Train 1.0000 | Val 0.7920 | Test 0.7940
Epoch 100 | Loss 0.1343 | Train 1.0000 | Val 0.7820 | Test 0.7910
Epoch 110 | Loss 0.1210 | Train 1.0000 | Val 0.7900 | Test 0.7890
Epoch 120 | Loss 0.1095 | Train 1.0000 | Val 0.7900 | Test 0.7920
Epoch 130 | Loss 0.0954 | Train 1.0000 | Val 0.7860 | Test 0.7900
Epoch 140 | Loss 0.1026 | Train 1.0000 | Val 0.7840 | Test 0.7930
Epoch 150 | Loss 0.0945 | Train 1.0000 | Val 0.7880 | Test 0.7920
Epoch 160 

## GAT Baseline

Cora

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

cora = Planetoid(
    root="data/Planetoid",
    name="Cora"
)

Coradata = cora[0].to(device)

gat_base_cora = GAT(
    in_channels=cora.num_features,
    hidden_channels=16,
    out_channels=cora.num_classes
).to(device)

optimizer = torch.optim.Adam(
    gat_base_cora.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(gat_base_cora,Coradata)

    train_acc, val_acc, test_acc = evaluate(gat_base_cora,Coradata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.8114 | Train 0.9643 | Val 0.7920 | Test 0.8080
Epoch 020 | Loss 0.6770 | Train 0.9929 | Val 0.8000 | Test 0.8190
Epoch 030 | Loss 0.6182 | Train 1.0000 | Val 0.7840 | Test 0.8030
Epoch 040 | Loss 0.5063 | Train 1.0000 | Val 0.7860 | Test 0.8130
Epoch 050 | Loss 0.4248 | Train 1.0000 | Val 0.7880 | Test 0.8090
Epoch 060 | Loss 0.4360 | Train 1.0000 | Val 0.7940 | Test 0.8030
Epoch 070 | Loss 0.3654 | Train 1.0000 | Val 0.7820 | Test 0.8020
Epoch 080 | Loss 0.3695 | Train 1.0000 | Val 0.7880 | Test 0.8160
Epoch 090 | Loss 0.4871 | Train 1.0000 | Val 0.7720 | Test 0.7890
Epoch 100 | Loss 0.4894 | Train 1.0000 | Val 0.7780 | Test 0.8030
Epoch 110 | Loss 0.4538 | Train 1.0000 | Val 0.7900 | Test 0.8160
Epoch 120 | Loss 0.4553 | Train 1.0000 | Val 0.7820 | Test 0.7960
Epoch 130 | Loss 0.3499 | Train 1.0000 | Val 0.7940 | Test 0.8000
Epoch 140 | Loss 0.3236 | Train 1.0000 | Val 0.7940 | Test 0.7990
Epoch 150 | Loss 0.3413 | Train 1.0000 | Val 0.7780 | Test 0.7970
Epoch 160 

CiteSeer

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

citeseer = Planetoid(
    root="data/Planetoid",
    name="CiteSeer"
)

citeseerdata = citeseer[0].to(device)

gat_base_citeseer = GAT(
    in_channels=citeseer.num_features,
    hidden_channels=16,
    out_channels=citeseer.num_classes
).to(device)

optimizer = torch.optim.Adam(
    gat_base_citeseer.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(gat_base_citeseer,citeseerdata)

    train_acc, val_acc, test_acc = evaluate(gat_base_citeseer,citeseerdata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.9805 | Train 0.9583 | Val 0.6900 | Test 0.6850
Epoch 020 | Loss 0.6753 | Train 0.9750 | Val 0.6840 | Test 0.6760
Epoch 030 | Loss 0.6246 | Train 1.0000 | Val 0.6700 | Test 0.6740
Epoch 040 | Loss 0.5504 | Train 1.0000 | Val 0.6660 | Test 0.6670
Epoch 050 | Loss 0.6171 | Train 1.0000 | Val 0.6740 | Test 0.6740
Epoch 060 | Loss 0.5193 | Train 1.0000 | Val 0.6720 | Test 0.6720
Epoch 070 | Loss 0.4640 | Train 0.9917 | Val 0.6580 | Test 0.6450
Epoch 080 | Loss 0.4958 | Train 1.0000 | Val 0.6540 | Test 0.6700
Epoch 090 | Loss 0.4917 | Train 0.9917 | Val 0.6520 | Test 0.6660
Epoch 100 | Loss 0.8129 | Train 1.0000 | Val 0.6460 | Test 0.6510
Epoch 110 | Loss 0.5042 | Train 1.0000 | Val 0.6460 | Test 0.6300
Epoch 120 | Loss 0.5903 | Train 1.0000 | Val 0.6680 | Test 0.6700
Epoch 130 | Loss 0.5385 | Train 1.0000 | Val 0.6640 | Test 0.6620
Epoch 140 | Loss 0.5233 | Train 0.9917 | Val 0.6540 | Test 0.6440
Epoch 150 | Loss 0.4978 | Train 1.0000 | Val 0.6500 | Test 0.6480
Epoch 160 

PubMed

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

pubmed = Planetoid(
    root="data/Planetoid",
    name="PubMed"
)

pubmeddata = pubmed[0].to(device)

gat_base_pubmed = GAT(
    in_channels=pubmed.num_features,
    hidden_channels=16,
    out_channels=pubmed.num_classes
).to(device)

optimizer = torch.optim.Adam(
    gat_base_pubmed.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(gat_base_pubmed,pubmeddata)

    train_acc, val_acc, test_acc = evaluate(gat_base_pubmed,pubmeddata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.2310 | Train 1.0000 | Val 0.7580 | Test 0.7560
Epoch 020 | Loss 0.3725 | Train 1.0000 | Val 0.7720 | Test 0.7750
Epoch 030 | Loss 0.2509 | Train 1.0000 | Val 0.7780 | Test 0.7760
Epoch 040 | Loss 0.1567 | Train 1.0000 | Val 0.7640 | Test 0.7630
Epoch 050 | Loss 0.2694 | Train 1.0000 | Val 0.7680 | Test 0.7730
Epoch 060 | Loss 0.4003 | Train 1.0000 | Val 0.7700 | Test 0.7790
Epoch 070 | Loss 0.2592 | Train 1.0000 | Val 0.7760 | Test 0.7700
Epoch 080 | Loss 0.2941 | Train 1.0000 | Val 0.7780 | Test 0.7620
Epoch 090 | Loss 0.3284 | Train 1.0000 | Val 0.7880 | Test 0.7660
Epoch 100 | Loss 0.2948 | Train 1.0000 | Val 0.7820 | Test 0.7710
Epoch 110 | Loss 0.3087 | Train 1.0000 | Val 0.7740 | Test 0.7680
Epoch 120 | Loss 0.2986 | Train 1.0000 | Val 0.7980 | Test 0.7770
Epoch 130 | Loss 0.2902 | Train 1.0000 | Val 0.7900 | Test 0.7740
Epoch 140 | Loss 0.3560 | Train 1.0000 | Val 0.7640 | Test 0.7590
Epoch 150 | Loss 0.3052 | Train 1.0000 | Val 0.7980 | Test 0.7790
Epoch 160 

## GraphSage Baseline

Cora

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

cora = Planetoid(
    root="data/Planetoid",
    name="Cora"
)

Coradata = cora[0].to(device)

graphsage_base_cora = GraphSAGE(
    in_channels=cora.num_features,
    hidden_channels=16,
    out_channels=cora.num_classes
).to(device)

optimizer = torch.optim.Adam(
    graphsage_base_cora.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(graphsage_base_cora,Coradata)

    train_acc, val_acc, test_acc = evaluate(graphsage_base_cora,Coradata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.4015 | Train 1.0000 | Val 0.7480 | Test 0.7630
Epoch 020 | Loss 0.0849 | Train 1.0000 | Val 0.7440 | Test 0.7800
Epoch 030 | Loss 0.0229 | Train 1.0000 | Val 0.7440 | Test 0.7730
Epoch 040 | Loss 0.0372 | Train 1.0000 | Val 0.7480 | Test 0.7790
Epoch 050 | Loss 0.0180 | Train 1.0000 | Val 0.7320 | Test 0.7780
Epoch 060 | Loss 0.0143 | Train 1.0000 | Val 0.7360 | Test 0.7820
Epoch 070 | Loss 0.0378 | Train 1.0000 | Val 0.7420 | Test 0.7770
Epoch 080 | Loss 0.0185 | Train 1.0000 | Val 0.7380 | Test 0.7770
Epoch 090 | Loss 0.0149 | Train 1.0000 | Val 0.7400 | Test 0.7810
Epoch 100 | Loss 0.0084 | Train 1.0000 | Val 0.7500 | Test 0.7970
Epoch 110 | Loss 0.0260 | Train 1.0000 | Val 0.7520 | Test 0.8000
Epoch 120 | Loss 0.0406 | Train 1.0000 | Val 0.7480 | Test 0.7940
Epoch 130 | Loss 0.0160 | Train 1.0000 | Val 0.7460 | Test 0.8000
Epoch 140 | Loss 0.0154 | Train 1.0000 | Val 0.7500 | Test 0.7870
Epoch 150 | Loss 0.0169 | Train 1.0000 | Val 0.7560 | Test 0.8020
Epoch 160 

CiteSeer

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

citeseer = Planetoid(
    root="data/Planetoid",
    name="CiteSeer"
)

citeseerdata = citeseer[0].to(device)

graphsage_base_citeseer = GraphSAGE(
    in_channels=citeseer.num_features,
    hidden_channels=16,
    out_channels=citeseer.num_classes
).to(device)

optimizer = torch.optim.Adam(
    graphsage_base_citeseer.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(graphsage_base_citeseer,citeseerdata)

    train_acc, val_acc, test_acc = evaluate(graphsage_base_citeseer,citeseerdata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.1517 | Train 1.0000 | Val 0.6520 | Test 0.6460
Epoch 020 | Loss 0.0125 | Train 1.0000 | Val 0.6680 | Test 0.6580
Epoch 030 | Loss 0.0206 | Train 1.0000 | Val 0.6720 | Test 0.6690
Epoch 040 | Loss 0.0164 | Train 1.0000 | Val 0.6440 | Test 0.6610
Epoch 050 | Loss 0.0197 | Train 1.0000 | Val 0.6560 | Test 0.6740
Epoch 060 | Loss 0.0372 | Train 1.0000 | Val 0.6600 | Test 0.6760
Epoch 070 | Loss 0.0417 | Train 1.0000 | Val 0.6720 | Test 0.6810
Epoch 080 | Loss 0.0142 | Train 1.0000 | Val 0.6760 | Test 0.6660
Epoch 090 | Loss 0.0224 | Train 1.0000 | Val 0.6700 | Test 0.6730
Epoch 100 | Loss 0.0226 | Train 1.0000 | Val 0.6740 | Test 0.6800
Epoch 110 | Loss 0.0228 | Train 1.0000 | Val 0.6800 | Test 0.6890
Epoch 120 | Loss 0.0072 | Train 1.0000 | Val 0.6700 | Test 0.6890
Epoch 130 | Loss 0.0131 | Train 1.0000 | Val 0.6680 | Test 0.6920
Epoch 140 | Loss 0.0154 | Train 1.0000 | Val 0.6660 | Test 0.6840
Epoch 150 | Loss 0.0062 | Train 1.0000 | Val 0.6680 | Test 0.6820
Epoch 160 

PubMed

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

pubmed = Planetoid(
    root="data/Planetoid",
    name="PubMed"
)

pubmeddata = pubmed[0].to(device)

graphsage_base_pubmed = GraphSAGE(
    in_channels=pubmed.num_features,
    hidden_channels=16,
    out_channels=pubmed.num_classes
).to(device)

optimizer = torch.optim.Adam(
    graphsage_base_pubmed.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

In [ ]:
best_val = 0
best_test = 0

for epoch in range(1, 201):

    loss = train(graphsage_base_pubmed,pubmeddata)

    train_acc, val_acc, test_acc = evaluate(graphsage_base_pubmed,pubmeddata)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print()
print(f"Best Validation: {best_val:.4f}")
print(f"Corresponding Test: {best_test:.4f}")

Epoch 010 | Loss 0.7809 | Train 0.9667 | Val 0.7560 | Test 0.7330
Epoch 020 | Loss 0.3665 | Train 0.9667 | Val 0.7660 | Test 0.7370
Epoch 030 | Loss 0.1628 | Train 1.0000 | Val 0.7660 | Test 0.7560
Epoch 040 | Loss 0.0975 | Train 1.0000 | Val 0.7800 | Test 0.7450
Epoch 050 | Loss 0.0528 | Train 1.0000 | Val 0.8020 | Test 0.7630
Epoch 060 | Loss 0.0549 | Train 1.0000 | Val 0.7900 | Test 0.7640
Epoch 070 | Loss 0.0437 | Train 1.0000 | Val 0.7820 | Test 0.7520
Epoch 080 | Loss 0.0402 | Train 1.0000 | Val 0.7880 | Test 0.7720
Epoch 090 | Loss 0.0269 | Train 1.0000 | Val 0.8020 | Test 0.7640
Epoch 100 | Loss 0.0615 | Train 1.0000 | Val 0.7940 | Test 0.7730
Epoch 110 | Loss 0.0492 | Train 1.0000 | Val 0.7940 | Test 0.7690
Epoch 120 | Loss 0.0476 | Train 1.0000 | Val 0.7840 | Test 0.7710
Epoch 130 | Loss 0.0363 | Train 1.0000 | Val 0.7940 | Test 0.7620
Epoch 140 | Loss 0.0296 | Train 1.0000 | Val 0.7920 | Test 0.7710
Epoch 150 | Loss 0.0332 | Train 1.0000 | Val 0.7940 | Test 0.7630
Epoch 160 

## Variantes

In [ ]:



SAVE_DIR = "/content/drive/MyDrive/XAI_final"

os.makedirs(SAVE_DIR, exist_ok=True)

results = []

for dataset_name in ["Cora", "CiteSeer", "PubMed"]:

    print(f"\n{'='*50}")
    print(f"Dataset: {dataset_name}")
    print(f"{'='*50}")

    dataset = Planetoid(
        root="data/Planetoid",
        name=dataset_name
    )

    data = dataset[0].to(device)

    for model_name in ["GCN", "GAT", "GraphSAGE"]:

        print(f"\nModel: {model_name}")

        variants = generate_model_variants(model_name)

        for variant_id, config in enumerate(variants):

            model = build_model(
                model_name=model_name,
                in_channels=dataset.num_features,
                out_channels=dataset.num_classes,
                config=config
            ).to(device)

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=0.01,
                weight_decay=5e-4
            )

            best_val = 0
            best_test = 0

            for epoch in range(1, 201):

                loss = train(
                    model,
                    data,
                    optimizer
                )

                train_acc, val_acc, test_acc = evaluate(
                    model,
                    data
                )

                if val_acc > best_val:

                    best_val = val_acc
                    best_test = test_acc

            result = {
                "dataset": dataset_name,
                "model": model_name,
                "variant_id": variant_id,
                "hidden_channels": config["hidden_channels"],
                "dropout": config["dropout"],
                "best_val": best_val,
                "best_test": best_test
            }

            if "heads" in config:
                result["heads"] = config["heads"]

            results.append(result)

            print(
                f"Variant {variant_id} | "
                f"Val={best_val:.4f} | "
                f"Test={best_test:.4f}"
            )


Dataset: Cora

Model: GCN
Variant 0 | Val=0.7820 | Test=0.8040
Variant 1 | Val=0.7980 | Test=0.8180
Variant 2 | Val=0.7920 | Test=0.7860
Variant 3 | Val=0.7840 | Test=0.8080
Variant 4 | Val=0.7800 | Test=0.8090
Variant 5 | Val=0.7980 | Test=0.8010
Variant 6 | Val=0.7980 | Test=0.8040
Variant 7 | Val=0.8060 | Test=0.8210
Variant 8 | Val=0.7960 | Test=0.8200

Model: GAT
Variant 0 | Val=0.7920 | Test=0.8070
Variant 1 | Val=0.8100 | Test=0.8240
Variant 2 | Val=0.7960 | Test=0.8150
Variant 3 | Val=0.8060 | Test=0.8210
Variant 4 | Val=0.7780 | Test=0.7820
Variant 5 | Val=0.8140 | Test=0.8140
Variant 6 | Val=0.8060 | Test=0.7860
Variant 7 | Val=0.8080 | Test=0.8030
Variant 8 | Val=0.8080 | Test=0.8110
Variant 9 | Val=0.8000 | Test=0.8210
Variant 10 | Val=0.7900 | Test=0.7900
Variant 11 | Val=0.7940 | Test=0.7950

Model: GraphSAGE
Variant 0 | Val=0.7780 | Test=0.7820
Variant 1 | Val=0.7820 | Test=0.7910
Variant 2 | Val=0.7900 | Test=0.8060
Variant 3 | Val=0.7900 | Test=0.8100
Variant 4 | Val=

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="best_test",
    ascending=False
)

results_df.head()

,dataset,model,variant_id,hidden_channels,dropout,best_val,best_test,heads
10,Cora,GAT,1,8,0.6,0.810,0.824,4.0
7,Cora,GCN,7,64,0.5,0.806,0.821,NaN
12,Cora,GAT,3,8,0.6,0.806,0.821,8.0
18,Cora,GAT,9,32,0.6,0.800,0.821,4.0
8,Cora,GCN,8,64,0.7,0.796,0.820,NaN


In [ ]:
results_df.to_csv(
    f"{SAVE_DIR}/all_results.csv",
    index=False
)

In [ ]:
top_models = (
    results_df
    .sort_values("best_test", ascending=False)
    .groupby(
        ["dataset", "model"],
        group_keys=False
    )
    .head(3)
)

In [ ]:
TOP_DIR = "/content/drive/MyDrive/XAI_final/top_models"

In [ ]:
for _, row in top_models.iterrows():

    dataset_name = row["dataset"]
    model_name = row["model"]

    dataset = Planetoid(
        root="data/Planetoid",
        name=dataset_name
    )

    data = dataset[0].to(device)

    config = {
        "hidden_channels": int(
            row["hidden_channels"]
        ),
        "dropout": float(
            row["dropout"]
        )
    }

    if "heads" in row and not pd.isna(row["heads"]):
        config["heads"] = int(row["heads"])

    model = build_model(
        model_name=model_name,
        in_channels=dataset.num_features,
        out_channels=dataset.num_classes,
        config=config
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.01,
        weight_decay=5e-4
    )

    for epoch in range(200):

        train(
            model,
            data,
            optimizer
        )

    filename = (
        f"{dataset_name}_"
        f"{model_name}_"
        f"h{config['hidden_channels']}_"
        f"d{config['dropout']}"
    )

    if "heads" in config:
        filename += f"_heads{config['heads']}"

    torch.save(
        model.state_dict(),
        f"{TOP_DIR}/{filename}.pth"
    )

# Estabilidade

In [22]:
TOP_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

## Dados

In [ ]:
PLdataset = Planetoid(root='data/Planetoid', name='Cora')
cora = PLdataset[0]

In [ ]:
CSdataset = Planetoid(root='data/Planetoid',name='CiteSeer')
citeseer = CSdataset[0]


In [ ]:
MPdataset = Planetoid(root='data/Planetoid',name='PubMed')
pubmed = MPdataset[0]

## Apoio

In [ ]:


warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

warnings.filterwarnings(
    "ignore"
)

In [23]:
def load_saved_models(
    dataset_name,
    model_name
):

    files = [

        f for f in os.listdir(TOP_DIR)

        if f.startswith(
            f"{dataset_name}_{model_name}"
        )

    ]

    models = {}

    for file in files:

        models[file] = load_model_from_file(
            file,
            dataset_name,
            model_name
        )

    return models

In [24]:
def load_model_from_file(filename, dataset_name, model_type):

    config = {}
    if model_type == "GCN" or model_type == "GraphSAGE":
        match = re.search(
            fr'{dataset_name}_{model_type}_h(\d+)_d(\d+\.\d+)',
            filename
        )
        if match:
            config["hidden_channels"] = int(match.group(1))
            config["dropout"] = float(match.group(2))
        else:
            raise ValueError(f"Filename {filename} does not match {model_type} pattern.")
    elif model_type == "GAT":
        match = re.search(
            fr'{dataset_name}_{model_type}_h(\d+)_d(\d+\.\d+)_heads(\d+)',
            filename
        )
        if match:
            config["hidden_channels"] = int(match.group(1))
            config["dropout"] = float(match.group(2))
            config["heads"] = int(match.group(3))
        else:
            raise ValueError(f"Filename {filename} does not match {model_type} pattern.")
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    dataset = Planetoid(
        root="data/Planetoid",
        name=dataset_name
    )

    model = build_model(
        model_name=model_type,
        in_channels=dataset.num_features,
        out_channels=dataset.num_classes,
        config=config
    ).to(device)

    model.load_state_dict(
        torch.load(
            os.path.join(
                TOP_DIR,
                filename
            ),
            map_location=device
        )
    )

    model.eval()

    return model

In [25]:
def create_explainer(model):

    return Explainer(

        model=model,

        algorithm=GNNExplainer(
            epochs=100
        ),

        explanation_type="model",

        node_mask_type="attributes",

        edge_mask_type="object",

        model_config=dict(
            mode="multiclass_classification",
            task_level="node",
            return_type="log_probs"
        )
    )

In [26]:
def get_top_features(
    explanation,
    node_id,
    k=10
):

    scores = explanation.node_mask

    if scores.ndim == 2:

        scores = scores[node_id]

    top_features = (
        torch.argsort(
            scores,
            descending=True
        )[:k]
        .cpu()
        .tolist()
    )

    return top_features

In [27]:
def make_explanations(
    models,
    selected_nodes,
    data,
    k=10
):

    explanations = {}

    for model_name, model in models.items():

        print(
            f"Explicando {model_name}"
        )

        explainer = create_explainer(
            model
        )

        explanations[
            model_name
        ] = {}

        for node_id in tqdm(selected_nodes):

            explanation = explainer(

                x=data.x,

                edge_index=data.edge_index,

                index=node_id
            )

            top_features = get_top_features(

                explanation,

                node_id=node_id,

                k=k
            )

            explanations[
                model_name
            ][node_id] = top_features

    return explanations

In [41]:
def get_graphlime_top_features(
    feature_scores,
    k=10
):

    return np.argsort(
        np.abs(feature_scores)
    )[::-1][:k].tolist()

In [ ]:


def make_graphlime_explanations(
    models,
    selected_nodes,
    data,
    k=10
):

    explanations = {}

    for model_name, model in models.items():

        print(
            f"Explicando {model_name}"
        )

        graphlime = GraphLIME(
            model,
            hop=2,
            rho=0.1
        )

        explanations[
            model_name
        ] = {}

        for node_id in tqdm(selected_nodes):

            feature_scores = graphlime.explain_node(

                node_id,

                data.x,

                data.edge_index
            )

            top_features = get_graphlime_top_features(

                feature_scores,

                k=k
            )

            explanations[
                model_name
            ][node_id] = top_features

    return explanations

In [29]:
def stability(
    explanations,
    selected_nodes
):

    model_ids = list(
        explanations.keys()
    )

    if len(model_ids) < 2:

        raise ValueError(
            "São necessários pelo menos 2 modelos."
        )

    stability_results = []

    for node_id in selected_nodes:

        jaccards = []
        overlaps = []
        spearmans = []

        for model_a, model_b in combinations(
            model_ids,
            2
        ):

            features_a = explanations[
                model_a
            ][node_id]

            features_b = explanations[
                model_b
            ][node_id]

            score_jaccard = (
                jaccard_similarity(
                    features_a,
                    features_b
                )
            )

            score_overlap = (
                overlap_at_k(
                    features_a,
                    features_b
                )
            )

            score_spearman = (
                spearman_topk(
                    features_a,
                    features_b
                )
            )

            if np.isnan(
                score_spearman
            ):
                score_spearman = 0

            jaccards.append(
                score_jaccard
            )

            overlaps.append(
                score_overlap
            )

            spearmans.append(
                score_spearman
            )

        stability_results.append({

            "node": node_id,

            "mean_jaccard":
                np.mean(jaccards),

            "mean_overlap":
                np.mean(overlaps),

            "mean_spearman":
                np.mean(spearmans),

            "std_jaccard":
                np.std(jaccards),

            "std_overlap":
                np.std(overlaps),

            "std_spearman":
                np.std(spearmans)

        })

    return pd.DataFrame(
        stability_results
    )

In [ ]:


def save_explanations(
    explanations,
    dataset,
    model,
    explainer,
    save_dir
):

    save_dir = Path(save_dir)

    save_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    filename = (
        f"{dataset}_"
        f"{model}_"
        f"{explainer}.pkl"
    )

    filepath = save_dir / filename

    with open(
        filepath,
        "wb"
    ) as f:

        pickle.dump(
            explanations,
            f
        )

    print(
        f"Salvo em: {filepath}"
    )

In [ ]:


def load_explanations(
    dataset,
    model,
    explainer,
    save_dir
):

    filepath = (

        Path(save_dir)

        /

        f"{dataset}_{model}_{explainer}.pkl"
    )

    with open(
        filepath,
        "rb"
    ) as f:

        return pickle.load(f)

## Metricas

In [30]:
def jaccard_similarity(a, b):

    a = set(a)
    b = set(b)

    return len(a & b) / len(a | b)

In [31]:
def overlap_at_k(a, b):

    return len(
        set(a) & set(b)
    )

In [32]:


def spearman_topk(a, b):

    common = list(
        set(a) & set(b)
    )

    if len(common) < 2:
        return 0

    rank_a = [
        a.index(x)
        for x in common
    ]

    rank_b = [
        b.index(x)
        for x in common
    ]

    corr, _ = spearmanr(
        rank_a,
        rank_b
    )

    return corr

## seleção de nodes

In [33]:
SEED=42

Cora

In [34]:
test_nodes_cora = torch.where(
    PLdataset.test_mask
)[0]

random.seed(SEED)

selected_nodes_cora = random.sample(
    test_nodes_cora.tolist(),
    20
)

print(selected_nodes_cora)

[2362, 1822, 1733, 2467, 1989, 1958, 1936, 1850, 2462, 1812, 2400, 2466, 2621, 2266, 1797, 2312, 2140, 1740, 1738, 1803]


CiteSeer

In [35]:
test_nodes_citeseer = torch.where(
    CSdataset.test_mask
)[0]

random.seed(SEED)

selected_nodes_citeseer = random.sample(
    test_nodes_citeseer.tolist(),
    20
)

print(selected_nodes_citeseer)

[2972, 2427, 2337, 3079, 2596, 2565, 2542, 2455, 3074, 2417, 3010, 3078, 3235, 2875, 2401, 2921, 2748, 2344, 2342, 2408]


PubMed

In [36]:
test_nodes_pubmed = torch.where(
    MPdataset.test_mask
)[0]

random.seed(SEED)

selected_nodes_pubmed = random.sample(
    test_nodes_pubmed.tolist(),
    20
)

print(selected_nodes_pubmed)

[19371, 18831, 18742, 19476, 18998, 18967, 18945, 18859, 19471, 18821, 19409, 19475, 19630, 19275, 18806, 19321, 19149, 18749, 18747, 18812]


## GNNExplainer

### CORA

#### GCN

In [104]:
cora_gcn_models = load_saved_models(
    "Cora",
    "GCN"
)


In [105]:
cora_gcn_models

{'Cora_GCN_h16_d0.5.pth': GCN(
   (conv1): GCNConv(1433, 16)
   (conv2): GCNConv(16, 7)
 ),
 'Cora_GCN_h64_d0.5.pth': GCN(
   (conv1): GCNConv(1433, 64)
   (conv2): GCNConv(64, 7)
 ),
 'Cora_GCN_h64_d0.7.pth': GCN(
   (conv1): GCNConv(1433, 64)
   (conv2): GCNConv(64, 7)
 )}

In [106]:
explanations_cora_GCN = make_explanations(cora_gcn_models, selected_nodes_cora, cora)

Explicando Cora_GCN_h16_d0.5.pth


100%|██████████| 20/20 [02:46<00:00,  8.31s/it]


Explicando Cora_GCN_h64_d0.5.pth


100%|██████████| 20/20 [03:20<00:00, 10.00s/it]


Explicando Cora_GCN_h64_d0.7.pth


100%|██████████| 20/20 [03:10<00:00,  9.54s/it]


In [ ]:
stability_cora_GCN_df = stability(explanations_cora_GCN, selected_nodes_cora)
stability_cora_GCN_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2362,0.623932,7.666667,0.130952,0.060436,0.471405,0.334324
1,1822,0.337302,5.000000,0.361905,0.072955,0.816497,0.681069
2,1733,1.000000,10.000000,0.668687,0.000000,0.000000,0.143192
3,2467,0.405678,5.666667,-0.354762,0.118871,1.247219,0.317105
4,1989,1.000000,10.000000,0.991919,0.000000,0.000000,0.005714


In [ ]:
print(
    stability_cora_GCN_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.709903
mean_overlap     8.050000
mean_spearman    0.295631
dtype: float64


In [154]:
save_explanations(
    explanations=explanations_cora_GCN,
    dataset="Cora",
    model="GCN",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Cora_GCN_GNNExplainer.pkl


#### GAT

In [107]:
cora_GAT_models = load_saved_models(
    "Cora",
    "GAT"
)

In [108]:
cora_GAT_models

{'Cora_GAT_h32_d0.6_heads4.pth': GAT(
   (gat1): GATConv(1433, 32, heads=4)
   (gat2): GATConv(128, 7, heads=1)
 ),
 'Cora_GAT_h8_d0.6_heads4.pth': GAT(
   (gat1): GATConv(1433, 8, heads=4)
   (gat2): GATConv(32, 7, heads=1)
 ),
 'Cora_GAT_h8_d0.6_heads8.pth': GAT(
   (gat1): GATConv(1433, 8, heads=8)
   (gat2): GATConv(64, 7, heads=1)
 )}

In [109]:
explanations_cora_GAT = make_explanations(cora_GAT_models, selected_nodes_cora, cora)

Explicando Cora_GAT_h32_d0.6_heads4.pth


100%|██████████| 20/20 [03:58<00:00, 11.94s/it]


Explicando Cora_GAT_h8_d0.6_heads4.pth


100%|██████████| 20/20 [02:59<00:00,  8.96s/it]


Explicando Cora_GAT_h8_d0.6_heads8.pth


100%|██████████| 20/20 [03:02<00:00,  9.14s/it]


In [110]:
stability_cora_GAT_df = stability(explanations_cora_GAT, selected_nodes_cora)
stability_cora_GAT_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2362,0.344538,5.000000,0.680952,0.118841,1.414214,0.127953
1,1822,0.312792,4.666667,-0.304762,0.103940,1.247219,0.436488
2,1733,1.000000,10.000000,0.466667,0.000000,0.000000,0.052370
3,2467,0.444444,6.000000,0.346032,0.157135,1.414214,0.415321
4,1989,1.000000,10.000000,0.983838,0.000000,0.000000,0.005714


In [111]:
print(
    stability_cora_GAT_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.653455
mean_overlap     7.516667
mean_spearman    0.255072
dtype: float64


In [155]:
save_explanations(
    explanations=explanations_cora_GAT,
    dataset="Cora",
    model="GAT",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Cora_GAT_GNNExplainer.pkl


#### GraphSAGE

In [112]:
cora_GraphSAGE_models = load_saved_models(
    "Cora",
    "GraphSAGE"
)

In [113]:
cora_GraphSAGE_models

{'Cora_GraphSAGE_h32_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(1433, 32, aggr=mean)
   (conv2): SAGEConv(32, 7, aggr=mean)
 ),
 'Cora_GraphSAGE_h32_d0.7.pth': GraphSAGE(
   (conv1): SAGEConv(1433, 32, aggr=mean)
   (conv2): SAGEConv(32, 7, aggr=mean)
 ),
 'Cora_GraphSAGE_h64_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(1433, 64, aggr=mean)
   (conv2): SAGEConv(64, 7, aggr=mean)
 )}

In [114]:
explanations_cora_GraphSAGE = make_explanations(cora_GraphSAGE_models, selected_nodes_cora, cora)

Explicando Cora_GraphSAGE_h32_d0.3.pth


100%|██████████| 20/20 [06:40<00:00, 20.01s/it]


Explicando Cora_GraphSAGE_h32_d0.7.pth


100%|██████████| 20/20 [07:19<00:00, 21.99s/it]


Explicando Cora_GraphSAGE_h64_d0.3.pth


100%|██████████| 20/20 [07:44<00:00, 23.21s/it]


In [115]:
stability_cora_GraphSAGE_df = stability(explanations_cora_GraphSAGE, selected_nodes_cora)
stability_cora_GraphSAGE_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2362,0.767677,8.666667,0.065873,0.071425,0.471405,0.297612
1,1822,0.396825,5.666667,-0.023810,0.044896,0.471405,0.442063
2,1733,1.000000,10.000000,0.696970,0.000000,0.000000,0.159276
3,2467,0.717172,8.333333,0.085714,0.071425,0.471405,0.462779
4,1989,1.000000,10.000000,0.931313,0.000000,0.000000,0.030236


In [116]:
print(
    stability_cora_GraphSAGE_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.756975
mean_overlap     8.416667
mean_spearman    0.330289
dtype: float64


In [156]:
save_explanations(
    explanations=explanations_cora_GraphSAGE,
    dataset="Cora",
    model="GraphSAGE",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Cora_GraphSAGE_GNNExplainer.pkl


### CiteSeer

#### GCN

In [117]:
citeseer_gcn_models = load_saved_models(
    "CiteSeer",
    "GCN"
)


In [118]:
citeseer_gcn_models

{'CiteSeer_GCN_h64_d0.3.pth': GCN(
   (conv1): GCNConv(3703, 64)
   (conv2): GCNConv(64, 6)
 ),
 'CiteSeer_GCN_h64_d0.5.pth': GCN(
   (conv1): GCNConv(3703, 64)
   (conv2): GCNConv(64, 6)
 ),
 'CiteSeer_GCN_h64_d0.7.pth': GCN(
   (conv1): GCNConv(3703, 64)
   (conv2): GCNConv(64, 6)
 )}

In [119]:
explanations_citeseer_GCN = make_explanations(citeseer_gcn_models, selected_nodes_citeseer, citeseer)

Explicando CiteSeer_GCN_h64_d0.3.pth


100%|██████████| 20/20 [06:48<00:00, 20.42s/it]


Explicando CiteSeer_GCN_h64_d0.5.pth


100%|██████████| 20/20 [06:47<00:00, 20.39s/it]


Explicando CiteSeer_GCN_h64_d0.7.pth


100%|██████████| 20/20 [06:47<00:00, 20.40s/it]


In [120]:
stability_citeseer_GCN_df = stability(explanations_citeseer_GCN, selected_nodes_citeseer)
stability_citeseer_GCN_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2972,0.428571,6.000000,-0.371429,0.000000,0.000000,0.186628
1,2427,0.225490,3.666667,-0.166667,0.034662,0.471405,0.402768
2,2337,0.390374,5.000000,-0.005556,0.302506,2.828427,0.711068
3,3079,0.369048,5.333333,0.447619,0.084179,0.942809,0.273385
4,2596,0.337302,5.000000,-0.371429,0.072955,0.816497,0.168224


In [121]:
print(
    stability_citeseer_GCN_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.539733
mean_overlap     6.683333
mean_spearman    0.065054
dtype: float64


In [157]:
save_explanations(
    explanations=explanations_citeseer_GCN,
    dataset="Citeseer",
    model="GCN",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Citeseer_GCN_GNNExplainer.pkl


#### GAT

In [122]:
citeseer_GAT_models = load_saved_models(
    "CiteSeer",
    "GAT"
)

In [123]:
citeseer_GAT_models

{'CiteSeer_GAT_h32_d0.4_heads4.pth': GAT(
   (gat1): GATConv(3703, 32, heads=4)
   (gat2): GATConv(128, 6, heads=1)
 ),
 'CiteSeer_GAT_h32_d0.6_heads8.pth': GAT(
   (gat1): GATConv(3703, 32, heads=8)
   (gat2): GATConv(256, 6, heads=1)
 ),
 'CiteSeer_GAT_h8_d0.4_heads4.pth': GAT(
   (gat1): GATConv(3703, 8, heads=4)
   (gat2): GATConv(32, 6, heads=1)
 )}

In [124]:
explanations_citeseer_GAT = make_explanations(citeseer_GAT_models, selected_nodes_citeseer, citeseer)

Explicando CiteSeer_GAT_h32_d0.4_heads4.pth


100%|██████████| 20/20 [08:25<00:00, 25.29s/it]


Explicando CiteSeer_GAT_h32_d0.6_heads8.pth


100%|██████████| 20/20 [11:03<00:00, 33.19s/it]


Explicando CiteSeer_GAT_h8_d0.4_heads4.pth


100%|██████████| 20/20 [06:50<00:00, 20.52s/it]


In [125]:
stability_citeseer_GAT_df = stability(explanations_citeseer_GAT, selected_nodes_citeseer)
stability_citeseer_GAT_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2972,0.200980,3.333333,-0.766667,0.034662,0.471405,0.205480
1,2427,0.159701,2.666667,0.200000,0.081443,1.247219,0.588784
2,2337,0.538462,7.000000,-0.392857,0.000000,0.000000,0.154303
3,3079,0.200980,3.333333,0.033333,0.034662,0.471405,0.731817
4,2596,0.157407,2.666667,-0.266667,0.065473,0.942809,0.899383


In [126]:
print(
    stability_citeseer_GAT_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.325301
mean_overlap     4.683333
mean_spearman   -0.074841
dtype: float64


In [158]:
save_explanations(
    explanations=explanations_citeseer_GAT,
    dataset="Citeseer",
    model="GAT",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Citeseer_GAT_GNNExplainer.pkl


#### GraphSAGE

In [127]:
citeseer_GraphSAGE_models = load_saved_models(
    "CiteSeer",
    "GraphSAGE"
)

In [128]:
citeseer_GraphSAGE_models

{'CiteSeer_GraphSAGE_h32_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(3703, 32, aggr=mean)
   (conv2): SAGEConv(32, 6, aggr=mean)
 ),
 'CiteSeer_GraphSAGE_h64_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(3703, 64, aggr=mean)
   (conv2): SAGEConv(64, 6, aggr=mean)
 ),
 'CiteSeer_GraphSAGE_h64_d0.5.pth': GraphSAGE(
   (conv1): SAGEConv(3703, 64, aggr=mean)
   (conv2): SAGEConv(64, 6, aggr=mean)
 )}

In [129]:
explanations_citeseer_GraphSAGE = make_explanations(citeseer_GraphSAGE_models, selected_nodes_citeseer, citeseer)

Explicando CiteSeer_GraphSAGE_h32_d0.3.pth


100%|██████████| 20/20 [17:49<00:00, 53.47s/it]


Explicando CiteSeer_GraphSAGE_h64_d0.3.pth


100%|██████████| 20/20 [17:57<00:00, 53.90s/it]


Explicando CiteSeer_GraphSAGE_h64_d0.5.pth


100%|██████████| 20/20 [19:07<00:00, 57.39s/it]


In [130]:
stability_citeseer_GraphSAGE_df = stability(explanations_citeseer_GraphSAGE, selected_nodes_citeseer)
stability_citeseer_GraphSAGE_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2972,0.428571,6.000000,-0.257143,0.000000,0.000000,0.491561
1,2427,0.211988,3.333333,0.366667,0.117706,1.699673,0.329983
2,2337,0.439394,5.666667,-0.188889,0.267843,2.357023,0.490150
3,3079,0.501832,6.666667,-0.326190,0.051803,0.471405,0.487822
4,2596,0.433455,6.000000,0.052381,0.083814,0.816497,0.392503


In [131]:
print(
    stability_citeseer_GraphSAGE_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.517244
mean_overlap     6.516667
mean_spearman    0.042868
dtype: float64


In [159]:
save_explanations(
    explanations=explanations_citeseer_GraphSAGE,
    dataset="Citeseer",
    model="GraphSAGE",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Citeseer_GraphSAGE_GNNExplainer.pkl


### PubmMed

#### GCN

In [132]:
pubmed_gcn_models = load_saved_models(
    "PubMed",
    "GCN"
)


In [133]:
pubmed_gcn_models

{'PubMed_GCN_h16_d0.7.pth': GCN(
   (conv1): GCNConv(500, 16)
   (conv2): GCNConv(16, 3)
 ),
 'PubMed_GCN_h32_d0.3.pth': GCN(
   (conv1): GCNConv(500, 32)
   (conv2): GCNConv(32, 3)
 ),
 'PubMed_GCN_h32_d0.5.pth': GCN(
   (conv1): GCNConv(500, 32)
   (conv2): GCNConv(32, 3)
 )}

In [134]:
explanations_pubmed_GCN = make_explanations(pubmed_gcn_models, selected_nodes_pubmed, pubmed)

Explicando PubMed_GCN_h16_d0.7.pth


100%|██████████| 20/20 [05:59<00:00, 17.95s/it]


Explicando PubMed_GCN_h32_d0.3.pth


100%|██████████| 20/20 [06:54<00:00, 20.74s/it]


Explicando PubMed_GCN_h32_d0.5.pth


100%|██████████| 20/20 [06:58<00:00, 20.90s/it]


In [135]:
stability_pubmed_GCN_df = stability(explanations_pubmed_GCN, selected_nodes_pubmed)
stability_pubmed_GCN_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,19371,0.250000,4.000000,0.666667,0.000000,0.000000,0.339935
1,18831,0.337302,5.000000,-0.476190,0.072955,0.816497,0.088320
2,18742,0.396825,5.666667,-0.209524,0.044896,0.471405,0.148156
3,19476,0.222222,2.666667,-0.206349,0.314270,3.771236,0.291822
4,18998,0.369048,5.333333,0.123810,0.084179,0.942809,0.679869


In [136]:
print(
    stability_pubmed_GCN_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.275256
mean_overlap     4.166667
mean_spearman   -0.030794
dtype: float64


In [160]:
save_explanations(
    explanations=explanations_pubmed_GCN,
    dataset="Pubmed",
    model="GCN",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Pubmed_GCN_GNNExplainer.pkl


#### GAT

In [137]:
pubmed_GAT_models = load_saved_models(
    "PubMed",
    "GAT"
)

In [138]:
pubmed_GAT_models

{'PubMed_GAT_h16_d0.4_heads4.pth': GAT(
   (gat1): GATConv(500, 16, heads=4)
   (gat2): GATConv(64, 3, heads=1)
 ),
 'PubMed_GAT_h32_d0.6_heads8.pth': GAT(
   (gat1): GATConv(500, 32, heads=8)
   (gat2): GATConv(256, 3, heads=1)
 ),
 'PubMed_GAT_h8_d0.6_heads8.pth': GAT(
   (gat1): GATConv(500, 8, heads=8)
   (gat2): GATConv(64, 3, heads=1)
 )}

In [139]:
explanations_pubmed_GAT = make_explanations(pubmed_GAT_models, selected_nodes_pubmed, pubmed)

Explicando PubMed_GAT_h16_d0.4_heads4.pth


100%|██████████| 20/20 [10:10<00:00, 30.55s/it]


Explicando PubMed_GAT_h32_d0.6_heads8.pth


100%|██████████| 20/20 [21:16<00:00, 63.83s/it]


Explicando PubMed_GAT_h8_d0.6_heads8.pth


100%|██████████| 20/20 [10:53<00:00, 32.68s/it]


In [140]:
stability_pubmed_GAT_df = stability(explanations_pubmed_GAT, selected_nodes_pubmed)
stability_pubmed_GAT_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,19371,0.135191,2.333333,0.666667,0.058378,0.942809,0.471405
1,18831,0.225490,3.666667,0.233333,0.034662,0.471405,0.543650
2,18742,0.305556,4.666667,-0.066667,0.039284,0.471405,0.518545
3,19476,0.128655,2.000000,0.200000,0.146316,2.160247,0.282843
4,18998,0.253268,4.000000,0.166667,0.064081,0.816497,0.235702


In [141]:
print(
    stability_pubmed_GAT_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.234776
mean_overlap     3.666667
mean_spearman    0.097857
dtype: float64


In [161]:
save_explanations(
    explanations=explanations_pubmed_GAT,
    dataset="Pubmed",
    model="GAT",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Pubmed_GAT_GNNExplainer.pkl


#### GraphSAGE

In [142]:
pubmed_GraphSAGE_models = load_saved_models(
    "PubMed",
    "GraphSAGE"
)

In [143]:
pubmed_GraphSAGE_models

{'PubMed_GraphSAGE_h16_d0.5.pth': GraphSAGE(
   (conv1): SAGEConv(500, 16, aggr=mean)
   (conv2): SAGEConv(16, 3, aggr=mean)
 ),
 'PubMed_GraphSAGE_h32_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(500, 32, aggr=mean)
   (conv2): SAGEConv(32, 3, aggr=mean)
 ),
 'PubMed_GraphSAGE_h64_d0.5.pth': GraphSAGE(
   (conv1): SAGEConv(500, 64, aggr=mean)
   (conv2): SAGEConv(64, 3, aggr=mean)
 )}

In [144]:
explanations_pubmed_GraphSAGE = make_explanations(pubmed_GraphSAGE_models, selected_nodes_pubmed, pubmed)

Explicando PubMed_GraphSAGE_h16_d0.5.pth


100%|██████████| 20/20 [19:54<00:00, 59.74s/it]


Explicando PubMed_GraphSAGE_h32_d0.3.pth


100%|██████████| 20/20 [20:30<00:00, 61.50s/it]


Explicando PubMed_GraphSAGE_h64_d0.5.pth


100%|██████████| 20/20 [22:55<00:00, 68.77s/it]


In [145]:
stability_pubmed_GraphSAGE_df = stability(explanations_pubmed_GraphSAGE, selected_nodes_pubmed)
stability_pubmed_GraphSAGE_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,19371,0.238718,3.666667,-0.471429,0.136873,1.699673,0.443701
1,18831,0.346154,5.000000,-0.400000,0.135982,1.414214,0.326599
2,18742,0.346154,5.000000,0.373810,0.135982,1.414214,0.328692
3,19476,0.401709,5.666667,0.138095,0.096698,0.942809,0.574081
4,18998,0.179194,3.000000,-0.033333,0.056734,0.816497,0.684755


In [146]:
print(
    stability_pubmed_GraphSAGE_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.259558
mean_overlap     3.983333
mean_spearman   -0.024881
dtype: float64


In [162]:
save_explanations(
    explanations=explanations_pubmed_GraphSAGE,
    dataset="Pubmed",
    model="GraphSAGE",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Pubmed_GraphSAGE_GNNExplainer.pkl


In [147]:
all_results = {

    "Cora_GCN":
        stability_cora_GCN_df,

    "Cora_GAT":
        stability_cora_GAT_df,

    "Cora_GraphSAGE":
        stability_cora_GraphSAGE_df,


    "CiteSeer_GCN":
        stability_citeseer_GCN_df,  

    "CiteSeer_GAT":
        stability_citeseer_GAT_df,
    "CiteSeer_GraphSAGE":
        stability_citeseer_GraphSAGE_df,
    "PubMed_GCN":
        stability_pubmed_GCN_df,    
    "PubMed_GAT":
        stability_pubmed_GAT_df,
    "PubMed_GraphSAGE":     
        stability_pubmed_GraphSAGE_df
}
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\all_stability_results.pkl",
    "wb"
) as f:

    pickle.dump(
        all_results,
        f
    )

NameError: name 'stability_cora_GCN_df' is not defined

In [ ]:
all_results = {

    "Cora_GCN":
        stability_cora_GCN_df,

    "Cora_GAT":
        stability_cora_GAT_df,

    "Cora_GraphSAGE":
        stability_cora_GraphSAGE_df,


    "CiteSeer_GCN":
        stability_citeseer_GCN_df,  

    "CiteSeer_GAT":
        stability_citeseer_GAT_df,
    "CiteSeer_GraphSAGE":
        stability_citeseer_GraphSAGE_df,
    "PubMed_GCN":
        stability_pubmed_GCN_df,    
    "PubMed_GAT":
        stability_pubmed_GAT_df,
    "PubMed_GraphSAGE":     
        stability_pubmed_GraphSAGE_df
}
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\all_stability_results.pkl",
    "wb"
) as f:

    pickle.dump(
        all_results,
        f
    )

## GraphLIME

### CORA

#### GCN

In [46]:
cora_gcn_models = load_saved_models(
    "Cora",
    "GCN"
)


In [47]:
cora_gcn_models

{'Cora_GCN_h16_d0.5.pth': GCN(
   (conv1): GCNConv(1433, 16)
   (conv2): GCNConv(16, 7)
 ),
 'Cora_GCN_h64_d0.5.pth': GCN(
   (conv1): GCNConv(1433, 64)
   (conv2): GCNConv(64, 7)
 ),
 'Cora_GCN_h64_d0.7.pth': GCN(
   (conv1): GCNConv(1433, 64)
   (conv2): GCNConv(64, 7)
 )}

In [48]:
explanations_graphlime_cora_GCN = make_graphlime_explanations(cora_gcn_models, selected_nodes_cora, cora)

Explicando Cora_GCN_h16_d0.5.pth


100%|██████████| 20/20 [00:23<00:00,  1.19s/it]


Explicando Cora_GCN_h64_d0.5.pth


100%|██████████| 20/20 [00:23<00:00,  1.16s/it]


Explicando Cora_GCN_h64_d0.7.pth


100%|██████████| 20/20 [00:47<00:00,  2.36s/it]


In [49]:
stability_graphlime_cora_GCN_df = stability(explanations_graphlime_cora_GCN, selected_nodes_cora)
stability_graphlime_cora_GCN_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2362,0.507937,6.666667,0.006349,0.112239,0.942809,0.138305
1,1822,0.137914,2.333333,0.666667,0.082774,1.247219,0.471405
2,1733,0.507937,6.666667,1.000000,0.112239,0.942809,0.000000
3,2467,0.631702,7.666667,0.500000,0.131861,0.942809,0.182108
4,1989,0.433455,6.000000,0.704762,0.083814,0.816497,0.110246


In [50]:
print(
    stability_graphlime_cora_GCN_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.529492
mean_overlap     6.416667
mean_spearman    0.572835
dtype: float64


In [163]:
save_explanations(
    explanations=explanations_graphlime_cora_GCN,
    dataset="Cora",
    model="GCN",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Cora_GCN_GraphLIME.pkl


#### GAT

In [51]:
cora_GAT_models = load_saved_models(
    "Cora",
    "GAT"
)

In [52]:
cora_GAT_models

{'Cora_GAT_h32_d0.6_heads4.pth': GAT(
   (gat1): GATConv(1433, 32, heads=4)
   (gat2): GATConv(128, 7, heads=1)
 ),
 'Cora_GAT_h8_d0.6_heads4.pth': GAT(
   (gat1): GATConv(1433, 8, heads=4)
   (gat2): GATConv(32, 7, heads=1)
 ),
 'Cora_GAT_h8_d0.6_heads8.pth': GAT(
   (gat1): GATConv(1433, 8, heads=8)
   (gat2): GATConv(64, 7, heads=1)
 )}

In [53]:
explanations_graphlime_cora_GAT = make_graphlime_explanations(cora_GAT_models, selected_nodes_cora, cora)

Explicando Cora_GAT_h32_d0.6_heads4.pth


100%|██████████| 20/20 [00:33<00:00,  1.66s/it]


Explicando Cora_GAT_h8_d0.6_heads4.pth


100%|██████████| 20/20 [00:31<00:00,  1.58s/it]


Explicando Cora_GAT_h8_d0.6_heads8.pth


100%|██████████| 20/20 [00:29<00:00,  1.45s/it]


In [54]:
stability_graphlime_cora_GAT_df = stability(explanations_graphlime_cora_GAT, selected_nodes_cora)
stability_graphlime_cora_GAT_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2362,1.000000,10.000000,1.0,0.000000,0.000000,0.000000
1,1822,0.113404,2.000000,0.5,0.050583,0.816497,0.408248
2,1733,1.000000,10.000000,1.0,0.000000,0.000000,0.000000
3,2467,1.000000,10.000000,1.0,0.000000,0.000000,0.000000
4,1989,0.200980,3.333333,1.0,0.034662,0.471405,0.000000


In [55]:
print(
    stability_graphlime_cora_GAT_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.622244
mean_overlap     6.800000
mean_spearman    0.709524
dtype: float64


In [164]:
save_explanations(
    explanations=explanations_graphlime_cora_GAT,
    dataset="Cora",
    model="GAT",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Cora_GAT_GraphLIME.pkl


#### GraphSAGE

In [56]:
cora_GraphSAGE_models = load_saved_models(
    "Cora",
    "GraphSAGE"
)

In [57]:
cora_GraphSAGE_models

{'Cora_GraphSAGE_h32_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(1433, 32, aggr=mean)
   (conv2): SAGEConv(32, 7, aggr=mean)
 ),
 'Cora_GraphSAGE_h32_d0.7.pth': GraphSAGE(
   (conv1): SAGEConv(1433, 32, aggr=mean)
   (conv2): SAGEConv(32, 7, aggr=mean)
 ),
 'Cora_GraphSAGE_h64_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(1433, 64, aggr=mean)
   (conv2): SAGEConv(64, 7, aggr=mean)
 )}

In [58]:
explanations_graphlime_cora_GraphSAGE = make_graphlime_explanations(cora_GraphSAGE_models, selected_nodes_cora, cora)

Explicando Cora_GraphSAGE_h32_d0.3.pth


100%|██████████| 20/20 [00:22<00:00,  1.12s/it]


Explicando Cora_GraphSAGE_h32_d0.7.pth


100%|██████████| 20/20 [00:19<00:00,  1.01it/s]


Explicando Cora_GraphSAGE_h64_d0.3.pth


100%|██████████| 20/20 [00:23<00:00,  1.16s/it]


In [59]:
stability_graphlime_cora_GraphSAGE_df = stability(explanations_graphlime_cora_GraphSAGE, selected_nodes_cora)
stability_graphlime_cora_GraphSAGE_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2362,0.619048,7.333333,0.428571,0.269374,1.885618,0.404061
1,1822,0.074074,1.333333,0.000000,0.052378,0.942809,0.816497
2,1733,0.818182,9.000000,1.000000,0.000000,0.000000,0.000000
3,2467,0.476190,6.333333,0.741270,0.140187,1.247219,0.185911
4,1989,0.200980,3.333333,1.000000,0.034662,0.471405,0.000000


In [60]:
print(
    stability_graphlime_cora_GraphSAGE_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.490932
mean_overlap     5.900000
mean_spearman    0.494596
dtype: float64


In [ ]:
save_explanations(
    explanations=explanations_graphlime_cora_GraphSAGE,
    dataset="Cora",
    model="GraphSAGE",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Cora_GraphSAGE_GraphLIME.pkl


### CiteSeer

#### GCN

In [61]:
citeseer_gcn_models = load_saved_models(
    "CiteSeer",
    "GCN"
)


In [62]:
citeseer_gcn_models

{'CiteSeer_GCN_h64_d0.3.pth': GCN(
   (conv1): GCNConv(3703, 64)
   (conv2): GCNConv(64, 6)
 ),
 'CiteSeer_GCN_h64_d0.5.pth': GCN(
   (conv1): GCNConv(3703, 64)
   (conv2): GCNConv(64, 6)
 ),
 'CiteSeer_GCN_h64_d0.7.pth': GCN(
   (conv1): GCNConv(3703, 64)
   (conv2): GCNConv(64, 6)
 )}

In [63]:
explanations_graphlime_citeseer_GCN = make_graphlime_explanations(citeseer_gcn_models, selected_nodes_citeseer, citeseer)

Explicando CiteSeer_GCN_h64_d0.3.pth


100%|██████████| 20/20 [00:09<00:00,  2.21it/s]


Explicando CiteSeer_GCN_h64_d0.5.pth


100%|██████████| 20/20 [00:07<00:00,  2.59it/s]


Explicando CiteSeer_GCN_h64_d0.7.pth


100%|██████████| 20/20 [00:07<00:00,  2.54it/s]


In [64]:
stability_graphlime_citeseer_GCN_df = stability(explanations_graphlime_citeseer_GCN, selected_nodes_citeseer)
stability_graphlime_citeseer_GCN_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2972,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
1,2427,0.076367,1.333333,0.166667,0.073973,1.247219,0.235702
2,2337,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
3,3079,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
4,2596,0.157407,2.666667,-0.333333,0.065473,0.942809,0.942809


In [65]:
print(
    stability_graphlime_citeseer_GCN_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.568002
mean_overlap     6.516667
mean_spearman    0.463373
dtype: float64


In [166]:
save_explanations(
    explanations=explanations_graphlime_citeseer_GCN,
    dataset="Citeseer",
    model="GCN",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Citeseer_GCN_GraphLIME.pkl


#### GAT

In [66]:
citeseer_GAT_models = load_saved_models(
    "CiteSeer",
    "GAT"
)

In [67]:
citeseer_GAT_models

{'CiteSeer_GAT_h32_d0.4_heads4.pth': GAT(
   (gat1): GATConv(3703, 32, heads=4)
   (gat2): GATConv(128, 6, heads=1)
 ),
 'CiteSeer_GAT_h32_d0.6_heads8.pth': GAT(
   (gat1): GATConv(3703, 32, heads=8)
   (gat2): GATConv(256, 6, heads=1)
 ),
 'CiteSeer_GAT_h8_d0.4_heads4.pth': GAT(
   (gat1): GATConv(3703, 8, heads=4)
   (gat2): GATConv(32, 6, heads=1)
 )}

In [68]:
explanations_graphlime_citeseer_GAT = make_graphlime_explanations(citeseer_GAT_models, selected_nodes_citeseer, citeseer)

Explicando CiteSeer_GAT_h32_d0.4_heads4.pth


100%|██████████| 20/20 [00:15<00:00,  1.27it/s]


Explicando CiteSeer_GAT_h32_d0.6_heads8.pth


100%|██████████| 20/20 [00:15<00:00,  1.33it/s]


Explicando CiteSeer_GAT_h8_d0.4_heads4.pth


100%|██████████| 20/20 [00:09<00:00,  2.19it/s]


In [69]:
stability_graphlime_citeseer_GAT_df = stability(explanations_graphlime_citeseer_GAT, selected_nodes_citeseer)
stability_graphlime_citeseer_GAT_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2972,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
1,2427,0.017544,0.333333,0.000000,0.024811,0.471405,0.000000
2,2337,1.000000,10.000000,0.773737,0.000000,0.000000,0.159992
3,3079,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
4,2596,0.157407,2.666667,0.266667,0.065473,0.942809,0.899383


In [70]:
print(
    stability_graphlime_citeseer_GAT_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.754143
mean_overlap     8.016667
mean_spearman    0.694654
dtype: float64


In [167]:
save_explanations(
    explanations=explanations_graphlime_citeseer_GAT,
    dataset="Citeseer",
    model="GAT",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Citeseer_GAT_GraphLIME.pkl


#### GraphSAGE

In [71]:
citeseer_GraphSAGE_models = load_saved_models(
    "CiteSeer",
    "GraphSAGE"
)

In [72]:
citeseer_GraphSAGE_models

{'CiteSeer_GraphSAGE_h32_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(3703, 32, aggr=mean)
   (conv2): SAGEConv(32, 6, aggr=mean)
 ),
 'CiteSeer_GraphSAGE_h64_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(3703, 64, aggr=mean)
   (conv2): SAGEConv(64, 6, aggr=mean)
 ),
 'CiteSeer_GraphSAGE_h64_d0.5.pth': GraphSAGE(
   (conv1): SAGEConv(3703, 64, aggr=mean)
   (conv2): SAGEConv(64, 6, aggr=mean)
 )}

In [73]:
explanations_graphlime_citeseer_GraphSAGE = make_graphlime_explanations(citeseer_GraphSAGE_models, selected_nodes_citeseer, citeseer)

Explicando CiteSeer_GraphSAGE_h32_d0.3.pth


100%|██████████| 20/20 [00:08<00:00,  2.32it/s]


Explicando CiteSeer_GraphSAGE_h64_d0.3.pth


100%|██████████| 20/20 [00:07<00:00,  2.60it/s]


Explicando CiteSeer_GraphSAGE_h64_d0.5.pth


100%|██████████| 20/20 [00:06<00:00,  3.09it/s]


In [74]:
stability_graphlime_citeseer_GraphSAGE_df = stability(explanations_graphlime_citeseer_GraphSAGE, selected_nodes_citeseer)
stability_graphlime_citeseer_GraphSAGE_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,2972,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
1,2427,0.093911,1.666667,0.166667,0.058378,0.942809,0.235702
2,2337,0.878788,9.333333,0.988889,0.085710,0.471405,0.007857
3,3079,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
4,2596,0.253268,4.000000,0.300000,0.064081,0.816497,0.616441


In [75]:
print(
    stability_graphlime_citeseer_GraphSAGE_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.572126
mean_overlap     6.566667
mean_spearman    0.457374
dtype: float64


In [168]:
save_explanations(
    explanations=explanations_graphlime_citeseer_GraphSAGE,
    dataset="Citeseer",
    model="GraphSAGE",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Citeseer_GraphSAGE_GraphLIME.pkl


### PubmMed

#### GCN

In [76]:
pubmed_gcn_models = load_saved_models(
    "PubMed",
    "GCN"
)


In [77]:
pubmed_gcn_models

{'PubMed_GCN_h16_d0.7.pth': GCN(
   (conv1): GCNConv(500, 16)
   (conv2): GCNConv(16, 3)
 ),
 'PubMed_GCN_h32_d0.3.pth': GCN(
   (conv1): GCNConv(500, 32)
   (conv2): GCNConv(32, 3)
 ),
 'PubMed_GCN_h32_d0.5.pth': GCN(
   (conv1): GCNConv(500, 32)
   (conv2): GCNConv(32, 3)
 )}

In [78]:
explanations_graphlime_pubmed_GCN = make_graphlime_explanations(pubmed_gcn_models, selected_nodes_pubmed, pubmed)

Explicando PubMed_GCN_h16_d0.7.pth


100%|██████████| 20/20 [00:18<00:00,  1.07it/s]


Explicando PubMed_GCN_h32_d0.3.pth


100%|██████████| 20/20 [00:15<00:00,  1.26it/s]


Explicando PubMed_GCN_h32_d0.5.pth


100%|██████████| 20/20 [00:22<00:00,  1.11s/it]


In [79]:
stability_graphlime_pubmed_GCN_df = stability(explanations_graphlime_pubmed_GCN, selected_nodes_pubmed)
stability_graphlime_pubmed_GCN_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,19371,0.581197,7.333333,0.599206,0.060436,0.471405,0.221726
1,18831,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
2,18742,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
3,19476,0.558442,7.000000,-0.135714,0.183664,1.414214,0.437176
4,18998,0.390374,5.000000,-0.022222,0.302506,2.828427,0.675680


In [82]:
print(
    stability_graphlime_pubmed_GCN_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.471433
mean_overlap     5.683333
mean_spearman    0.497143
dtype: float64


In [169]:
save_explanations(
    explanations=explanations_graphlime_pubmed_GCN,
    dataset="Pubmed",
    model="GCN",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Pubmed_GCN_GraphLIME.pkl


#### GAT

In [83]:
pubmed_GAT_models = load_saved_models(
    "PubMed",
    "GAT"
)

In [84]:
pubmed_GAT_models

{'PubMed_GAT_h16_d0.4_heads4.pth': GAT(
   (gat1): GATConv(500, 16, heads=4)
   (gat2): GATConv(64, 3, heads=1)
 ),
 'PubMed_GAT_h32_d0.6_heads8.pth': GAT(
   (gat1): GATConv(500, 32, heads=8)
   (gat2): GATConv(256, 3, heads=1)
 ),
 'PubMed_GAT_h8_d0.6_heads8.pth': GAT(
   (gat1): GATConv(500, 8, heads=8)
   (gat2): GATConv(64, 3, heads=1)
 )}

In [85]:
explanations_graphlime_pubmed_GAT = make_graphlime_explanations(pubmed_GAT_models, selected_nodes_pubmed, pubmed)

Explicando PubMed_GAT_h16_d0.4_heads4.pth


100%|██████████| 20/20 [00:19<00:00,  1.00it/s]


Explicando PubMed_GAT_h32_d0.6_heads8.pth


100%|██████████| 20/20 [00:16<00:00,  1.23it/s]


Explicando PubMed_GAT_h8_d0.6_heads8.pth


100%|██████████| 20/20 [00:15<00:00,  1.32it/s]


In [86]:
stability_graphlime_pubmed_GAT_df = stability(explanations_graphlime_pubmed_GAT, selected_nodes_pubmed)
stability_graphlime_pubmed_GAT_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,19371,0.132898,2.333333,-0.333333,0.030811,0.471405,0.942809
1,18831,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
2,18742,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000
3,19476,0.595072,7.333333,0.764286,0.164017,1.247219,0.166701
4,18998,0.238718,3.666667,0.052381,0.136873,1.699673,0.746906


In [87]:
print(
    stability_graphlime_pubmed_GAT_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.409584
mean_overlap     4.833333
mean_spearman    0.518333
dtype: float64


In [170]:
save_explanations(
    explanations=explanations_graphlime_pubmed_GAT,
    dataset="Pubmed",
    model="GAT",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Pubmed_GAT_GraphLIME.pkl


#### GraphSAGE

In [88]:
pubmed_GraphSAGE_models = load_saved_models(
    "PubMed",
    "GraphSAGE"
)

In [89]:
pubmed_GraphSAGE_models

{'PubMed_GraphSAGE_h16_d0.5.pth': GraphSAGE(
   (conv1): SAGEConv(500, 16, aggr=mean)
   (conv2): SAGEConv(16, 3, aggr=mean)
 ),
 'PubMed_GraphSAGE_h32_d0.3.pth': GraphSAGE(
   (conv1): SAGEConv(500, 32, aggr=mean)
   (conv2): SAGEConv(32, 3, aggr=mean)
 ),
 'PubMed_GraphSAGE_h64_d0.5.pth': GraphSAGE(
   (conv1): SAGEConv(500, 64, aggr=mean)
   (conv2): SAGEConv(64, 3, aggr=mean)
 )}

In [90]:
explanations_graphlime_pubmed_GraphSAGE = make_graphlime_explanations(pubmed_GraphSAGE_models, selected_nodes_pubmed, pubmed)

Explicando PubMed_GraphSAGE_h16_d0.5.pth


100%|██████████| 20/20 [00:16<00:00,  1.20it/s]


Explicando PubMed_GraphSAGE_h32_d0.3.pth


100%|██████████| 20/20 [00:11<00:00,  1.72it/s]


Explicando PubMed_GraphSAGE_h64_d0.5.pth


100%|██████████| 20/20 [00:11<00:00,  1.72it/s]


In [91]:
stability_graphlime_pubmed_GraphSAGE_df = stability(explanations_graphlime_pubmed_GraphSAGE, selected_nodes_pubmed)
stability_graphlime_pubmed_GraphSAGE_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,19371,0.118421,2.0,-0.133333,0.093040,1.414214,0.188562
1,18831,1.000000,10.0,1.000000,0.000000,0.000000,0.000000
2,18742,1.000000,10.0,1.000000,0.000000,0.000000,0.000000
3,19476,0.558442,7.0,0.531746,0.183664,1.414214,0.165156
4,18998,0.260504,4.0,0.814286,0.118841,1.414214,0.223455


In [92]:
print(
    stability_graphlime_pubmed_GraphSAGE_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.467492
mean_overlap     5.483333
mean_spearman    0.583532
dtype: float64


In [171]:
save_explanations(
    explanations=explanations_graphlime_pubmed_GraphSAGE,
    dataset="Pubmed",
    model="GraphSAGE",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\Pubmed_GraphSAGE_GraphLIME.pkl


## Resultados

In [93]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\all_stability_results.pkl",
    "rb"
) as f:

    all_results_gnnexplainer = pickle.load(f)

In [94]:
import pandas as pd

summary_rows = []

for experiment_name, df in all_results_gnnexplainer.items():

    dataset, model = experiment_name.split("_")

    summary_rows.append({

        "Dataset": dataset,

        "Modelo": model,

        "Explicador": "GNNExplainer",

        "Jaccard":
            df["mean_jaccard"].mean(),

        "Overlap@10":
            df["mean_overlap"].mean(),

        "Spearman":
            df["mean_spearman"].mean(),

        "Jaccard Std":
            df["mean_jaccard"].std()

    })

summary_df = pd.DataFrame(
    summary_rows
)

summary_df

,Dataset,Modelo,Explicador,Jaccard,Overlap@10,Spearman,Jaccard Std
0,Cora,GCN,GNNExplainer,0.709903,8.050000,0.295631,0.242612
1,Cora,GAT,GNNExplainer,0.654203,7.516667,0.314639,0.286925
2,Cora,GraphSAGE,GNNExplainer,0.760395,8.466667,0.415094,0.206014
3,CiteSeer,GCN,GNNExplainer,0.511074,6.383333,0.003622,0.236919
4,CiteSeer,GAT,GNNExplainer,0.337440,4.866667,-0.152341,0.133163
5,CiteSeer,GraphSAGE,GNNExplainer,0.537195,6.716667,0.018110,0.193718
6,PubMed,GCN,GNNExplainer,0.305415,4.466667,-0.029206,0.125980
7,PubMed,GAT,GNNExplainer,0.237350,3.650000,0.063373,0.110112
8,PubMed,GraphSAGE,GNNExplainer,0.271284,4.100000,0.026984,0.103597


In [95]:
all_results_graphlime = {

    "Cora_GCN":
        stability_graphlime_cora_GCN_df,

    "Cora_GAT":
        stability_graphlime_cora_GAT_df,

    "Cora_GraphSAGE":
        stability_graphlime_cora_GraphSAGE_df,

    "CiteSeer_GCN":
        stability_graphlime_citeseer_GCN_df,

    "CiteSeer_GAT":
        stability_graphlime_citeseer_GAT_df,

    "CiteSeer_GraphSAGE":   
        stability_graphlime_citeseer_GraphSAGE_df,

    "PubMed_GCN":
        stability_graphlime_pubmed_GCN_df,
        
    "PubMed_GAT":
        stability_graphlime_pubmed_GAT_df,
        
    "PubMed_GraphSAGE":
        stability_graphlime_pubmed_GraphSAGE_df
}

In [96]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\all_stability_results_graphlime.pkl",
    "wb"
) as f:

    pickle.dump(
        all_results_graphlime,
        f
    )

In [98]:
summary_rows = []

for explainer_name, results in {

    "GNNExplainer":
        all_results_gnnexplainer,

    "GraphLIME":
        all_results_graphlime

}.items():

    for experiment_name, df in results.items():

        dataset, model = experiment_name.split("_")

        summary_rows.append({

            "Dataset": dataset,

            "Modelo": model,

            "Explicador": explainer_name,

            "Jaccard":
                df["mean_jaccard"].mean(),

            "Overlap@10":
                df["mean_overlap"].mean(),

            "Spearman":
                df["mean_spearman"].mean(),

            "Jaccard Std":
                df["mean_jaccard"].std()

        })

estabilidade_summary_df = pd.DataFrame(
    summary_rows
)

estabilidade_summary_df

,Dataset,Modelo,Explicador,Jaccard,Overlap@10,Spearman,Jaccard Std
0,Cora,GCN,GNNExplainer,0.709903,8.050000,0.295631,0.242612
1,Cora,GAT,GNNExplainer,0.654203,7.516667,0.314639,0.286925
2,Cora,GraphSAGE,GNNExplainer,0.760395,8.466667,0.415094,0.206014
3,CiteSeer,GCN,GNNExplainer,0.511074,6.383333,0.003622,0.236919
4,CiteSeer,GAT,GNNExplainer,0.337440,4.866667,-0.152341,0.133163
5,CiteSeer,GraphSAGE,GNNExplainer,0.537195,6.716667,0.018110,0.193718
6,PubMed,GCN,GNNExplainer,0.305415,4.466667,-0.029206,0.125980
7,PubMed,GAT,GNNExplainer,0.237350,3.650000,0.063373,0.110112
8,PubMed,GraphSAGE,GNNExplainer,0.271284,4.100000,0.026984,0.103597
9,Cora,GCN,GraphLIME,0.529492,6.416667,0.572835,0.289410


In [100]:
estabilidade_summary_df = estabilidade_summary_df.sort_values(

    [
        "Dataset",
        "Modelo",
        "Explicador"
    ]
)

estabilidade_summary_df

,Dataset,Modelo,Explicador,Jaccard,Overlap@10,Spearman,Jaccard Std
4,CiteSeer,GAT,GNNExplainer,0.337440,4.866667,-0.152341,0.133163
13,CiteSeer,GAT,GraphLIME,0.754143,8.016667,0.694654,0.317082
3,CiteSeer,GCN,GNNExplainer,0.511074,6.383333,0.003622,0.236919
12,CiteSeer,GCN,GraphLIME,0.568002,6.516667,0.463373,0.361509
5,CiteSeer,GraphSAGE,GNNExplainer,0.537195,6.716667,0.018110,0.193718
14,CiteSeer,GraphSAGE,GraphLIME,0.572126,6.566667,0.457374,0.358043
1,Cora,GAT,GNNExplainer,0.654203,7.516667,0.314639,0.286925
10,Cora,GAT,GraphLIME,0.622244,6.800000,0.709524,0.397092
0,Cora,GCN,GNNExplainer,0.709903,8.050000,0.295631,0.242612
9,Cora,GCN,GraphLIME,0.529492,6.416667,0.572835,0.289410


In [99]:
estabilidade_summary_df.to_csv(
    r"C:\faculdade\Tabalho-Final-XAI\results\estabilidade_df.csv",
    index=False
)

# Fidelidade

In [ ]:
TOP_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

## Dados

In [ ]:
PLdataset = Planetoid(root='data/Planetoid', name='Cora')
cora = PLdataset[0]

In [ ]:
CSdataset = Planetoid(root='data/Planetoid',name='CiteSeer')
citeseer = CSdataset[0]


In [ ]:
MPdataset = Planetoid(root='data/Planetoid',name='PubMed')
pubmed = MPdataset[0]

## Apoio

In [ ]:


def fidelity_score(
    model,
    data,
    node_id,
    important_features
):

    model.eval()

    with torch.no_grad():

        logits = model(
            data.x,
            data.edge_index
        )

        probs = F.softmax(
            logits,
            dim=1
        )

        predicted_class = probs[
            node_id
        ].argmax().item()

        original_prob = probs[
            node_id,
            predicted_class
        ].item()

        x_masked = data.x.clone()

        x_masked[
            node_id,
            important_features
        ] = 0

        logits_masked = model(
            x_masked,
            data.edge_index
        )

        probs_masked = F.softmax(
            logits_masked,
            dim=1
        )

        masked_prob = probs_masked[
            node_id,
            predicted_class
        ].item()

        return (
            original_prob
            - masked_prob
        )

In [ ]:


def fidelity(
    models,
    explanations,
    selected_nodes,
    data
):

    fidelity_results = []

    for node_id in selected_nodes:

        scores = []

        for model_name, model in models.items():

            important_features = explanations[
                model_name
            ][node_id]

            score = fidelity_score(

                model,

                data,

                node_id,

                important_features
            )

            scores.append(score)

        fidelity_results.append({

            "node": node_id,

            "mean_fidelity":
                np.mean(scores),

            "std_fidelity":
                np.std(scores),

            "min_fidelity":
                np.min(scores),

            "max_fidelity":
                np.max(scores)

        })

    return pd.DataFrame(
        fidelity_results
    )

## seleção de nodes

In [178]:
SEED=42

Cora

In [179]:
test_nodes_cora = torch.where(
    PLdataset.test_mask
)[0]

random.seed(SEED)

selected_nodes_cora = random.sample(
    test_nodes_cora.tolist(),
    20
)

print(selected_nodes_cora)

[2362, 1822, 1733, 2467, 1989, 1958, 1936, 1850, 2462, 1812, 2400, 2466, 2621, 2266, 1797, 2312, 2140, 1740, 1738, 1803]


CiteSeer

In [180]:
test_nodes_citeseer = torch.where(
    CSdataset.test_mask
)[0]

random.seed(SEED)

selected_nodes_citeseer = random.sample(
    test_nodes_citeseer.tolist(),
    20
)

print(selected_nodes_citeseer)

[2972, 2427, 2337, 3079, 2596, 2565, 2542, 2455, 3074, 2417, 3010, 3078, 3235, 2875, 2401, 2921, 2748, 2344, 2342, 2408]


PubMed

In [181]:
test_nodes_pubmed = torch.where(
    MPdataset.test_mask
)[0]

random.seed(SEED)

selected_nodes_pubmed = random.sample(
    test_nodes_pubmed.tolist(),
    20
)

print(selected_nodes_pubmed)

[19371, 18831, 18742, 19476, 18998, 18967, 18945, 18859, 19471, 18821, 19409, 19475, 19630, 19275, 18806, 19321, 19149, 18749, 18747, 18812]


## GNNExplainer

### Cora

#### GCN

In [174]:
cora_gcn_models = load_saved_models(
    "Cora",
    "GCN"
)

In [175]:
cora_gcn_gnne_exp = load_explanations(
    dataset="Cora",
    model="GCN",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [182]:
fidelity_cora_gcn = fidelity(

    models=cora_gcn_models,

    explanations=cora_gcn_gnne_exp,

    selected_nodes=selected_nodes_cora,

    data=cora
)

In [183]:
fidelity_cora_gcn

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2362,0.343796,0.161325,0.117956,0.484750
1,1822,0.008682,0.004248,0.003131,0.013449
2,1733,0.074273,0.001197,0.072604,0.075351
3,2467,0.308652,0.040531,0.251632,0.342225
4,1989,-0.150838,0.019546,-0.174737,-0.126858
5,1958,0.000190,0.000014,0.000179,0.000210
6,1936,0.195655,0.009307,0.185433,0.207946
7,1850,0.023402,0.001891,0.021454,0.025963
8,2462,0.179483,0.073321,0.096232,0.274643
9,1812,0.011460,0.000668,0.010520,0.012011


#### GAT

In [184]:
cora_gat_models = load_saved_models(
    "Cora",
    "GAT"
)

In [185]:
cora_gat_gnne_exp = load_explanations(
    dataset="Cora",
    model="GAT",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [186]:
fidelity_cora_gat = fidelity(

    models=cora_gat_models,

    explanations=cora_gat_gnne_exp,

    selected_nodes=selected_nodes_cora,

    data=cora
)

In [187]:
fidelity_cora_gat

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2362,0.019385,0.004264,0.015475,0.025316
1,1822,0.005259,0.005706,0.000251,0.013242
2,1733,0.075027,0.074850,-0.030033,0.138766
3,2467,0.046222,0.041445,0.016486,0.104832
4,1989,-0.066321,0.023965,-0.084363,-0.032453
5,1958,0.000231,0.000172,0.000044,0.000459
6,1936,0.067207,0.046302,0.015803,0.128038
7,1850,0.011468,0.005684,0.006534,0.019430
8,2462,0.261229,0.035207,0.235173,0.311001
9,1812,-0.003941,0.002635,-0.006065,-0.000227


#### GraphSAGE

In [188]:
cora_graphsage_models = load_saved_models(
    "Cora",
    "GraphSAGE"
)

In [189]:
cora_graphsage_gnne_exp = load_explanations(
    dataset="Cora",
    model="GraphSAGE",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [190]:
fidelity_cora_graphsage = fidelity(

    models=cora_graphsage_models,

    explanations=cora_graphsage_gnne_exp,

    selected_nodes=selected_nodes_cora,

    data=cora
)

In [191]:
fidelity_cora_graphsage

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2362,0.059798,0.031631,0.029851,0.103549
1,1822,0.006349,0.004105,0.000915,0.010837
2,1733,-0.082820,0.052883,-0.150971,-0.022071
3,2467,0.489967,0.107833,0.397324,0.641192
4,1989,-0.140708,0.068297,-0.204518,-0.046010
5,1958,0.016082,0.011058,0.003104,0.030127
6,1936,0.386357,0.141209,0.186796,0.492565
7,1850,0.412964,0.022909,0.384549,0.440651
8,2462,0.110647,0.075506,0.053906,0.217358
9,1812,0.052135,0.012283,0.039118,0.068605


### CiteSeer

#### GCN

In [192]:
cite_seer_gcn_models = load_saved_models(
    "CiteSeer",
    "GCN"
)

In [193]:
cite_seer_gcn_gnne_exp = load_explanations(
    dataset="CiteSeer",
    model="GCN",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [194]:
fidelity_cite_seer_gcn = fidelity(

    models=cite_seer_gcn_models,

    explanations=cite_seer_gcn_gnne_exp,

    selected_nodes=selected_nodes_citeseer,

    data=citeseer
)

In [195]:
fidelity_cite_seer_gcn

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2972,0.418897,0.164880,0.217549,0.621416
1,2427,0.206780,0.045280,0.161493,0.268631
2,2337,0.419925,0.024500,0.398324,0.454187
3,3079,0.025437,0.009248,0.015071,0.037526
4,2596,0.003482,0.001027,0.002207,0.004723
5,2565,0.083228,0.049618,0.014485,0.129795
6,2542,0.540409,0.160612,0.313435,0.661427
7,2455,0.043555,0.007184,0.035315,0.052821
8,3074,0.453195,0.018424,0.431512,0.476549
9,2417,0.053145,0.014637,0.032788,0.066573


#### GAT

In [196]:
cite_seer_gat_models = load_saved_models(
    "CiteSeer",
    "GAT"
)

In [197]:
cite_seer_gat_gnne_exp = load_explanations(
    dataset="CiteSeer",
    model="GAT",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [198]:
fidelity_cite_seer_gat = fidelity(

    models=cite_seer_gat_models,

    explanations=cite_seer_gat_gnne_exp,

    selected_nodes=selected_nodes_citeseer,

    data=citeseer
)

In [199]:
fidelity_cite_seer_gat

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2972,0.131061,0.021651,0.114844,0.161662
1,2427,0.079846,0.048368,0.013159,0.126371
2,2337,0.379816,0.087843,0.267930,0.482511
3,3079,0.034571,0.017742,0.018399,0.059271
4,2596,0.001069,0.000656,0.000258,0.001864
5,2565,0.012193,0.007234,0.004189,0.021714
6,2542,0.038590,0.027767,0.006470,0.074214
7,2455,0.003039,0.001078,0.001664,0.004297
8,3074,0.034057,0.026276,0.005636,0.069000
9,2417,0.000387,0.000198,0.000111,0.000561


#### GraphSAGE

In [200]:
citeseer_graphsage_models = load_saved_models(
    "CiteSeer",
    "GraphSAGE"
)

In [201]:
citeseer_graphsage_gnne_exp = load_explanations(
    dataset="CiteSeer",
    model="GraphSAGE",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [202]:
fidelity_cite_seer_graphsage = fidelity(

    models=citeseer_graphsage_models,

    explanations=citeseer_graphsage_gnne_exp,

    selected_nodes=selected_nodes_citeseer,

    data=citeseer
)

In [203]:
fidelity_cite_seer_graphsage

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2972,0.313318,0.045071,0.275437,0.376653
1,2427,0.306416,0.019737,0.288769,0.333968
2,2337,0.321664,0.148352,0.136593,0.499779
3,3079,0.051300,0.021372,0.032928,0.081271
4,2596,0.127097,0.014128,0.108942,0.143398
5,2565,0.317986,0.087235,0.194696,0.383468
6,2542,0.075418,0.030027,0.037553,0.110995
7,2455,0.080791,0.015380,0.059199,0.093863
8,3074,0.023687,0.007772,0.016643,0.034516
9,2417,0.066405,0.001660,0.064618,0.068616


### PubMed

#### GCN

In [204]:
pubmed_gcn_models = load_saved_models(
    "PubMed",
    "GCN"
)

In [205]:
pubmed_gcn_gnne_exp = load_explanations(
    dataset="PubMed",
    model="GCN",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [206]:
fidelity_pubmed_gcn = fidelity(

    models=pubmed_gcn_models,

    explanations=pubmed_gcn_gnne_exp,

    selected_nodes=selected_nodes_pubmed,

    data=pubmed
)

In [207]:
fidelity_pubmed_gcn

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,19371,0.085134,0.043689,0.025265,0.128294
1,18831,0.003898,0.001689,0.001936,0.006059
2,18742,0.000126,0.000024,0.000109,0.000160
3,19476,0.123707,0.012257,0.106871,0.135700
4,18998,0.024446,0.004307,0.018358,0.027664
5,18967,0.016543,0.009202,0.009089,0.029509
6,18945,0.057039,0.004517,0.051505,0.062570
7,18859,0.013650,0.007185,0.006867,0.023593
8,19471,0.044117,0.007311,0.038858,0.054456
9,18821,0.009445,0.004405,0.003590,0.014218


#### GAT

In [208]:
pubmed_gat_models = load_saved_models(
    "PubMed",
    "GAT"
)

In [209]:
pubmed_gat_gnne_exp = load_explanations(
    dataset="PubMed",
    model="GAT",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [249]:
fidelity_pubmed_gat = fidelity(

    models=pubmed_gat_models,

    explanations=pubmed_gat_gnne_exp,

    selected_nodes=selected_nodes_pubmed,

    data=pubmed
)

In [250]:
fidelity_pubmed_gat

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,19371,0.042164,0.051844,0.004093,0.115465
1,18831,0.001391,0.000823,0.000455,0.002459
2,18742,0.005909,0.000822,0.004916,0.006928
3,19476,0.116371,0.013144,0.098650,0.130091
4,18998,0.022479,0.007643,0.016218,0.033240
5,18967,0.009979,0.001246,0.008244,0.011111
6,18945,0.044616,0.022819,0.025206,0.076648
7,18859,0.010593,0.005530,0.003009,0.016040
8,19471,0.037227,0.008407,0.030360,0.049065
9,18821,0.007302,0.008712,-0.001221,0.019268


#### GraphSAGE

In [212]:
pubmed_graphsage_models = load_saved_models(
    "PubMed",
    "GraphSAGE"
)

In [213]:
pubmed_graphsage_gnne_exp = load_explanations(
    dataset="PubMed",
    model="GraphSAGE",
    explainer="GNNExplainer",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [214]:
fidelity_pubmed_graphsage = fidelity(

    models=pubmed_graphsage_models,

    explanations=pubmed_graphsage_gnne_exp,

    selected_nodes=selected_nodes_pubmed,

    data=pubmed
)

In [215]:
fidelity_pubmed_graphsage

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,19371,0.054403,0.036439,0.003803,0.088154
1,18831,0.041336,0.026403,0.020810,0.078611
2,18742,0.016358,0.007287,0.006055,0.021713
3,19476,0.096798,0.026228,0.077656,0.133883
4,18998,0.006880,0.004112,0.003815,0.012692
5,18967,0.092042,0.035427,0.042865,0.124927
6,18945,0.108872,0.008715,0.100318,0.120833
7,18859,0.081345,0.061753,0.023772,0.167001
8,19471,0.062401,0.020874,0.034707,0.085100
9,18821,0.053313,0.037916,0.007547,0.100394


## GraphLIME

### Cora

#### GCN

In [216]:
cora_gcn_models = load_saved_models(
    "Cora",
    "GCN"
)

In [217]:
cora_gcn_graphlime_exp = load_explanations(
    dataset="Cora",
    model="GCN",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [218]:
fidelity_cora_gcn_graphlime = fidelity(

    models=cora_gcn_models,

    explanations=cora_gcn_graphlime_exp,

    selected_nodes=selected_nodes_cora,

    data=cora
)

In [219]:
fidelity_cora_gcn_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2362,-0.001035,0.001813,-0.003509,0.000782
1,1822,-0.000126,0.000118,-0.000293,-0.000034
2,1733,0.000000,0.000000,0.000000,0.000000
3,2467,0.126055,0.025236,0.091023,0.149474
4,1989,0.000000,0.000000,0.000000,0.000000
5,1958,0.000000,0.000000,0.000000,0.000000
6,1936,-0.002123,0.003002,-0.006368,0.000000
7,1850,0.000000,0.000000,0.000000,0.000000
8,2462,-0.020486,0.015886,-0.032612,0.001956
9,1812,0.000000,0.000000,0.000000,0.000000


#### GAT

In [220]:
cora_gat_models = load_saved_models(
    "Cora",
    "GAT"
)

In [221]:
cora_gat_graphlime_exp = load_explanations(
    dataset="Cora",
    model="GAT",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [222]:
fidelity_cora_gat_graphlime = fidelity(

    models=cora_gat_models,

    explanations=cora_gat_graphlime_exp,

    selected_nodes=selected_nodes_cora,

    data=cora
)

In [223]:
fidelity_cora_gat_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2362,-0.000154,0.000172,-0.000356,6.383657e-05
1,1822,-0.000012,0.000009,-0.000021,-2.384186e-07
2,1733,0.000000,0.000000,0.000000,0.000000e+00
3,2467,0.014378,0.014962,0.003565,3.553534e-02
4,1989,0.000000,0.000000,0.000000,0.000000e+00
5,1958,-0.000023,0.000024,-0.000057,-4.410744e-06
6,1936,0.001022,0.001446,0.000000,3.067434e-03
7,1850,0.000000,0.000000,0.000000,0.000000e+00
8,2462,-0.054448,0.010968,-0.065176,-3.938287e-02
9,1812,0.000000,0.000000,0.000000,0.000000e+00


#### GraphSAGE

In [224]:
cora_graphsage_models = load_saved_models(
    "Cora",
    "GraphSAGE"
)

In [225]:
cora_graphsage_graphlime_exp = load_explanations(
    dataset="Cora",
    model="GraphSAGE",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [226]:
fidelity_cora_graphsage_graphlime = fidelity(

    models=cora_graphsage_models,

    explanations=cora_graphsage_graphlime_exp,

    selected_nodes=selected_nodes_cora,

    data=cora
)

In [227]:
fidelity_cora_graphsage_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2362,2.384186e-07,0.000075,-0.000103,0.000073
1,1822,0.000000e+00,0.000000,0.000000,0.000000
2,1733,0.000000e+00,0.000000,0.000000,0.000000
3,2467,2.804271e-02,0.051366,-0.038315,0.086819
4,1989,0.000000e+00,0.000000,0.000000,0.000000
5,1958,0.000000e+00,0.000000,0.000000,0.000000
6,1936,-4.881223e-04,0.000690,-0.001464,0.000000
7,1850,2.825419e-03,0.003996,0.000000,0.008476
8,2462,3.085195e-02,0.038391,-0.018488,0.075143
9,1812,0.000000e+00,0.000000,0.000000,0.000000


### CiteSeer

#### GCN

In [228]:
cite_seer_gcn_models = load_saved_models(
    "CiteSeer",
    "GCN"
)

In [229]:
cite_seer_gcn_graphlime_exp = load_explanations(
    dataset="CiteSeer",
    model="GCN",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [230]:
fidelity_cite_seer_gcn_graphlime = fidelity(

    models=cite_seer_gcn_models,

    explanations=cite_seer_gcn_graphlime_exp,

    selected_nodes=selected_nodes_citeseer,

    data=citeseer
)

In [231]:
fidelity_cite_seer_gcn_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2972,0.000000,0.000000,0.000000,0.000000
1,2427,-0.000265,0.000375,-0.000796,0.000000
2,2337,0.031877,0.031903,0.007022,0.076914
3,3079,0.001711,0.000481,0.001050,0.002179
4,2596,0.000000,0.000000,0.000000,0.000000
5,2565,-0.004074,0.003916,-0.009496,-0.000385
6,2542,0.020243,0.027462,0.000248,0.059074
7,2455,-0.009012,0.006437,-0.014632,0.000000
8,3074,0.071443,0.118014,-0.092720,0.179576
9,2417,0.002115,0.002991,0.000000,0.006344


#### GAT

In [232]:
cite_seer_gat_models = load_saved_models(
    "CiteSeer",
    "GAT"
)

In [233]:
cite_seer_gat_graphlime_exp = load_explanations(
    dataset="CiteSeer",
    model="GAT",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [234]:
fidelity_cite_seer_gat_graphlime = fidelity(

    models=cite_seer_gat_models,

    explanations=cite_seer_gat_graphlime_exp,

    selected_nodes=selected_nodes_citeseer,

    data=citeseer
)

In [235]:
fidelity_cite_seer_gat_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2972,0.000000,0.000000,0.000000,0.000000
1,2427,0.000000,0.000000,0.000000,0.000000
2,2337,0.099728,0.054877,0.024691,0.154402
3,3079,0.005765,0.003836,0.001582,0.010848
4,2596,0.000000,0.000000,0.000000,0.000000
5,2565,-0.000517,0.000299,-0.000771,-0.000097
6,2542,-0.000421,0.000802,-0.001552,0.000215
7,2455,0.000115,0.000162,0.000000,0.000345
8,3074,0.001715,0.000865,0.001094,0.002939
9,2417,0.000021,0.000023,0.000000,0.000052


#### GraphSAGE

In [236]:
citeseer_graphsage_models = load_saved_models(
    "CiteSeer",
    "GraphSAGE"
)

In [237]:
citeseer_graphsage_graphlime_exp = load_explanations(
    dataset="CiteSeer",
    model="GraphSAGE",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [238]:
fidelity_cite_seer_graphsage_graphlime = fidelity(

    models=citeseer_graphsage_models,

    explanations=citeseer_graphsage_graphlime_exp,

    selected_nodes=selected_nodes_citeseer,

    data=citeseer
)

In [239]:
fidelity_cite_seer_graphsage_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,2972,0.000000,0.000000,0.000000,0.000000
1,2427,0.000000,0.000000,0.000000,0.000000
2,2337,0.024013,0.016389,0.006879,0.046097
3,3079,0.001760,0.000417,0.001383,0.002341
4,2596,-0.000444,0.000628,-0.001333,0.000000
5,2565,-0.006116,0.003719,-0.011306,-0.002783
6,2542,0.000807,0.001150,-0.000123,0.002427
7,2455,-0.000177,0.000250,-0.000531,0.000000
8,3074,-0.000469,0.001814,-0.002151,0.002049
9,2417,0.001588,0.002246,0.000000,0.004764


### PubMed

#### GCN

In [240]:
pubmed_gcn_models = load_saved_models(
    "PubMed",
    "GCN"
)

In [241]:
pubmed_gcn_graphlime_exp = load_explanations(
    dataset="PubMed",
    model="GCN",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [242]:
fidelity_pubmed_gcn_graphlime = fidelity(

    models=pubmed_gcn_models,

    explanations=pubmed_gcn_graphlime_exp,

    selected_nodes=selected_nodes_pubmed,

    data=pubmed
)

In [243]:
fidelity_pubmed_gcn_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,19371,1.419295e-01,9.002197e-03,1.310326e-01,1.530791e-01
1,18831,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,18742,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,19476,-3.309263e-02,5.827595e-02,-9.701633e-02,4.391864e-02
4,18998,-5.676746e-04,3.062077e-03,-4.868627e-03,2.019644e-03
5,18967,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
6,18945,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
7,18859,1.942515e-04,2.747132e-04,0.000000e+00,5.827546e-04
8,19471,-1.232326e-03,2.032681e-03,-4.060447e-03,6.278157e-04
9,18821,1.006315e-02,3.322491e-03,6.138027e-03,1.426256e-02


#### GAT

In [244]:
pubmed_gat_models = load_saved_models(
    "PubMed",
    "GAT"
)

In [245]:
pubmed_gat_graphlime_exp = load_explanations(
    dataset="PubMed",
    model="GAT",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [247]:
fidelity_pubmed_gat_graphlime = fidelity(

    models=pubmed_gat_models,

    explanations=pubmed_gat_graphlime_exp,

    selected_nodes=selected_nodes_pubmed,

    data=pubmed
)

In [248]:
fidelity_pubmed_gat_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,19371,0.000181,0.000718,-0.000594,0.001136
1,18831,0.000000,0.000000,0.000000,0.000000
2,18742,0.000000,0.000000,0.000000,0.000000
3,19476,0.009240,0.021014,-0.012565,0.037630
4,18998,0.000139,0.000684,-0.000829,0.000641
5,18967,0.000000,0.000000,0.000000,0.000000
6,18945,0.001581,0.002236,0.000000,0.004744
7,18859,0.000000,0.000000,0.000000,0.000000
8,19471,-0.001743,0.001935,-0.003794,0.000851
9,18821,0.007307,0.001961,0.004587,0.009136


#### GraphSAGE

In [251]:
pubmed_graphsage_models = load_saved_models(
    "PubMed",
    "GraphSAGE"
)

In [252]:
pubmed_graphsage_graphlime_exp = load_explanations(
    dataset="PubMed",
    model="GraphSAGE",
    explainer="GraphLIME",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [253]:
fidelity_pubmed_graphsage_graphlime = fidelity(

    models=pubmed_graphsage_models,

    explanations=pubmed_graphsage_graphlime_exp,

    selected_nodes=selected_nodes_pubmed,

    data=pubmed
)

In [254]:
fidelity_pubmed_graphsage_graphlime

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,19371,-3.080567e-04,2.289517e-04,-0.000548,0.000000
1,18831,0.000000e+00,0.000000e+00,0.000000,0.000000
2,18742,0.000000e+00,0.000000e+00,0.000000,0.000000
3,19476,-5.757131e-02,2.322289e-02,-0.086414,-0.029548
4,18998,8.532802e-04,1.381943e-03,-0.000243,0.002803
5,18967,0.000000e+00,0.000000e+00,0.000000,0.000000
6,18945,0.000000e+00,0.000000e+00,0.000000,0.000000
7,18859,0.000000e+00,0.000000e+00,0.000000,0.000000
8,19471,3.607273e-03,5.595357e-03,-0.000688,0.011510
9,18821,3.727901e-02,1.483281e-02,0.022506,0.057563


## Resultados

In [257]:
all_results_fidelity_gnnexplainer = {

    "Cora_GCN":
        fidelity_cora_gcn,

    "Cora_GAT":
        fidelity_cora_gat,

    "Cora_GraphSAGE":
        fidelity_cora_graphsage,

    "CiteSeer_GCN":
        fidelity_cite_seer_gcn,

    "CiteSeer_GAT":
        fidelity_cite_seer_gat,

    "CiteSeer_GraphSAGE":   
        fidelity_cite_seer_graphsage,

    "PubMed_GCN":
        fidelity_pubmed_gcn,
        
    "PubMed_GAT":
        fidelity_pubmed_gat,
        
    "PubMed_GraphSAGE":
        fidelity_pubmed_graphsage
}

In [256]:
all_results_fidelity_graphlime = {

    "Cora_GCN":
        fidelity_cora_gcn_graphlime,

    "Cora_GAT":
        fidelity_cora_gat_graphlime,

    "Cora_GraphSAGE":
        fidelity_cora_graphsage_graphlime,

    "CiteSeer_GCN":
        fidelity_cite_seer_gcn_graphlime,

    "CiteSeer_GAT":
        fidelity_cite_seer_gat_graphlime,

    "CiteSeer_GraphSAGE":   
        fidelity_cite_seer_graphsage_graphlime,

    "PubMed_GCN":
        fidelity_pubmed_gcn_graphlime,
        
    "PubMed_GAT":
        fidelity_pubmed_gat_graphlime,
        
    "PubMed_GraphSAGE":
        fidelity_pubmed_graphsage_graphlime
}

In [ ]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\all_stability_results_graphlime.pkl",
    "wb"
) as f:

    pickle.dump(
        all_results_graphlime,
        f
    )

In [258]:
summary_rows = []

for explainer_name, results in {

    "GNNExplainer":
        all_results_fidelity_gnnexplainer,

    "GraphLIME":
        all_results_fidelity_graphlime

}.items():

    for experiment_name, df in results.items():

        dataset, model = experiment_name.split("_")

        summary_rows.append({

            "Dataset": dataset,

            "Modelo": model,

            "Explicador": explainer_name,

            "Fidelity":
                df["mean_fidelity"].mean(),

            "Fidelity Std":
                df["mean_fidelity"].std(),

            "Min Fidelity":
                df["min_fidelity"].mean(),

            "Max Fidelity":
                df["max_fidelity"].mean()

        })

fidelity_summary_df = pd.DataFrame(
    summary_rows
)

fidelity_summary_df = fidelity_summary_df.sort_values(
    ["Dataset", "Modelo", "Explicador"]
)

fidelity_summary_df

,Dataset,Modelo,Explicador,Fidelity,Fidelity Std,Min Fidelity,Max Fidelity
4,CiteSeer,GAT,GNNExplainer,0.106221,0.138034,0.057383,0.157533
13,CiteSeer,GAT,GraphLIME,0.000713,0.026676,-0.010686,0.012836
3,CiteSeer,GCN,GNNExplainer,0.196855,0.180720,0.147629,0.241742
12,CiteSeer,GCN,GraphLIME,0.009755,0.021735,-0.003311,0.022673
5,CiteSeer,GraphSAGE,GNNExplainer,0.167530,0.154371,0.133833,0.206884
14,CiteSeer,GraphSAGE,GraphLIME,-0.000765,0.011293,-0.007926,0.011373
1,Cora,GAT,GNNExplainer,0.054330,0.113856,0.029242,0.081092
10,Cora,GAT,GraphLIME,-0.001016,0.013451,-0.003819,0.002857
0,Cora,GCN,GNNExplainer,0.093679,0.145885,0.065037,0.122685
9,Cora,GCN,GraphLIME,0.007342,0.029521,0.002095,0.014431


In [261]:
fidelity_summary_df.to_csv(
    r"C:\faculdade\Tabalho-Final-XAI\results\fidelity_global.csv",
    index=False
)

In [260]:
summary_rows = []

for explainer_name, results in {

    "GNNExplainer":
        all_results_fidelity_gnnexplainer,

    "GraphLIME":
        all_results_fidelity_graphlime

}.items():

    for experiment_name, df in results.items():

        dataset, model = experiment_name.split("_")

        for _, row in df.iterrows():

            summary_rows.append({

                "Dataset": dataset,

                "Modelo": model,

                "Explicador": explainer_name,

                "Node": row["node"],

                "Mean Fidelity":
                    row["mean_fidelity"],

                "Std Fidelity":
                    row["std_fidelity"],

                "Min Fidelity":
                    row["min_fidelity"],

                "Max Fidelity":
                    row["max_fidelity"]

            })

fidelity_node_df = pd.DataFrame(
    summary_rows
)

fidelity_node_df

,Dataset,Modelo,Explicador,Node,Mean Fidelity,Std Fidelity,Min Fidelity,Max Fidelity
0,Cora,GCN,GNNExplainer,2362.0,3.437956e-01,1.613248e-01,0.117956,0.484750
1,Cora,GCN,GNNExplainer,1822.0,8.681794e-03,4.248290e-03,0.003131,0.013449
2,Cora,GCN,GNNExplainer,1733.0,7.427290e-02,1.196991e-03,0.072604,0.075351
3,Cora,GCN,GNNExplainer,2467.0,3.086520e-01,4.053060e-02,0.251632,0.342225
4,Cora,GCN,GNNExplainer,1989.0,-1.508385e-01,1.954629e-02,-0.174737,-0.126858
...,...,...,...,...,...,...,...,...
355,PubMed,GraphSAGE,GraphLIME,19321.0,4.236182e-03,5.990865e-03,0.000000,0.012709
356,PubMed,GraphSAGE,GraphLIME,19149.0,-1.437384e-02,1.254046e-02,-0.030557,0.000000
357,PubMed,GraphSAGE,GraphLIME,18749.0,3.576279e-07,5.057622e-07,0.000000,0.000001
358,PubMed,GraphSAGE,GraphLIME,18747.0,-7.474105e-03,5.318695e-03,-0.011943,0.000000


In [262]:
fidelity_node_df.to_csv(
    r"C:\faculdade\Tabalho-Final-XAI\results\fidelity_node.csv",
    index=False
)